In [1]:
import json
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Project paths

PROJECT_ROOT = Path.cwd().parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "customer_support_tickets.csv"

SPLIT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "lumora_split_indices_v1.json"
)

# Load original dataset

df = pd.read_csv(DATA_PATH)

# Load saved split indices

with open(SPLIT_PATH, "r", encoding="utf-8") as file:
    split_indices = json.load(file)

print("Dataset shape:", df.shape)

print("Training records:", len(split_indices["train"]))
print("Validation records:", len(split_indices["validation"]))
print("Test records:", len(split_indices["test"]))

FileNotFoundError: [Errno 2] No such file or directory: '/Users/raghad/Desktop/Lumora/data/processed/lumora_split_indices_v1.json'

In [3]:
from pathlib import Path

PROJECT_ROOT = Path.home() / "Desktop" / "Lumora"

print("Project root exists:", PROJECT_ROOT.exists())

print("\nSearching for split file:")

for path in PROJECT_ROOT.rglob("lumora_split_indices_v1.json"):
    print(path)

Project root exists: True

Searching for split file:
/Users/raghad/Desktop/Lumora/notebooks/data/processed/lumora_split_indices_v1.json


In [4]:
import json
import pandas as pd
from pathlib import Path

# Project root
PROJECT_ROOT = Path.home() / "Desktop" / "Lumora"

# File paths
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "customer_support_tickets.csv"

SPLIT_PATH = (
    PROJECT_ROOT
    / "notebooks"
    / "data"
    / "processed"
    / "lumora_split_indices_v1.json"
)

# Load dataset
df = pd.read_csv(DATA_PATH)

# Load saved split indices
with open(SPLIT_PATH, "r", encoding="utf-8") as file:
    split_indices = json.load(file)

print("Dataset shape:", df.shape)
print("Training records:", len(split_indices["train"]))
print("Validation records:", len(split_indices["validation"]))
print("Test records:", len(split_indices["test"]))

Dataset shape: (26872, 5)
Training records: 18798
Validation records: 4041
Test records: 4033


In [5]:
# Reconstruct datasets using saved split indices

train_df = df.iloc[split_indices["train"]].copy()
val_df = df.iloc[split_indices["validation"]].copy()
test_df = df.iloc[split_indices["test"]].copy()

# Separate text features and target labels

X_train = train_df["instruction"]
y_train = train_df["category"]

X_val = val_df["instruction"]
y_val = val_df["category"]

X_test = test_df["instruction"]
y_test = test_df["category"]

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Training: (18798,) (18798,)
Validation: (4041,) (4041,)
Test: (4033,) (4033,)


In [6]:
# Verify dataset integrity

assert len(X_train) == len(y_train)
assert len(X_val) == len(y_val)
assert len(X_test) == len(y_test)

assert set(y_train.unique()) == set(y_val.unique())
assert set(y_train.unique()) == set(y_test.unique())

assert df["instruction"].notna().all()
assert df["category"].notna().all()

print("All dataset integrity checks passed.")

All dataset integrity checks passed.


In [7]:
# Normalize instructions for leakage verification

import re

def normalize_instruction(text):
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text


train_groups = set(X_train.apply(normalize_instruction))
val_groups = set(X_val.apply(normalize_instruction))
test_groups = set(X_test.apply(normalize_instruction))


# Check group overlap

print("Training ∩ Validation:", len(train_groups & val_groups))
print("Training ∩ Test:", len(train_groups & test_groups))
print("Validation ∩ Test:", len(val_groups & test_groups))

Training ∩ Validation: 0
Training ∩ Test: 0
Validation ∩ Test: 0


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

print("TF-IDF vectorizer configured successfully.")

TF-IDF vectorizer configured successfully.


In [9]:
# Learn vocabulary and transform training text

X_train_tfidf = tfidf.fit_transform(X_train)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Number of features:", X_train_tfidf.shape[1])
print("Number of documents:", X_train_tfidf.shape[0])

Training TF-IDF shape: (18798, 4876)
Number of features: 4876
Number of documents: 18798


In [10]:
# Transform validation data using the fitted TF-IDF vectorizer

X_val_tfidf = tfidf.transform(X_val)

print("Validation TF-IDF shape:", X_val_tfidf.shape)

Validation TF-IDF shape: (4041, 4876)


In [11]:
# Initialize the baseline Logistic Regression model

baseline_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# Train the model using training data only

baseline_model.fit(X_train_tfidf, y_train)

print("Baseline Logistic Regression trained successfully.")

Baseline Logistic Regression trained successfully.


In [12]:
# Generate predictions on validation data

y_val_pred = baseline_model.predict(X_val_tfidf)

print("Validation predictions generated successfully.")
print("Number of predictions:", len(y_val_pred))

Validation predictions generated successfully.
Number of predictions: 4041


In [13]:
# Calculate validation accuracy

val_accuracy = accuracy_score(y_val, y_val_pred)

print(f"Validation Accuracy: {val_accuracy:.4f}")
print(f"Validation Accuracy (%): {val_accuracy * 100:.2f}%")

Validation Accuracy: 0.9955
Validation Accuracy (%): 99.55%


In [14]:
# Evaluate precision, recall, and F1-score for each category

print(classification_report(y_val, y_val_pred))

              precision    recall  f1-score   support

     ACCOUNT       0.99      1.00      0.99       981
      CANCEL       0.99      1.00      1.00       150
     CONTACT       1.00      0.99      0.99       295
    DELIVERY       0.99      1.00      0.99       289
    FEEDBACK       0.99      1.00      1.00       275
     INVOICE       1.00      1.00      1.00       271
       ORDER       1.00      0.99      1.00       573
     PAYMENT       1.00      0.99      0.99       312
      REFUND       1.00      0.99      1.00       449
    SHIPPING       1.00      0.98      0.99       299
SUBSCRIPTION       1.00      1.00      1.00       147

    accuracy                           1.00      4041
   macro avg       1.00      1.00      1.00      4041
weighted avg       1.00      1.00      1.00      4041



In [15]:
# Generate confusion matrix

cm = confusion_matrix(
    y_val,
    y_val_pred,
    labels=baseline_model.classes_
)

cm_df = pd.DataFrame(
    cm,
    index=baseline_model.classes_,
    columns=baseline_model.classes_
)

cm_df

,ACCOUNT,CANCEL,CONTACT,DELIVERY,FEEDBACK,INVOICE,ORDER,PAYMENT,REFUND,SHIPPING,SUBSCRIPTION
ACCOUNT,981,0,0,0,0,0,0,0,0,0,0
CANCEL,0,150,0,0,0,0,0,0,0,0,0
CONTACT,2,0,292,0,0,0,0,0,0,1,0
DELIVERY,0,0,0,289,0,0,0,0,0,0,0
FEEDBACK,0,0,0,0,275,0,0,0,0,0,0
INVOICE,0,0,0,0,0,271,0,0,0,0,0
ORDER,2,0,0,1,0,0,570,0,0,0,0
PAYMENT,3,0,0,1,0,0,0,308,0,0,0
REFUND,0,1,0,0,2,0,0,0,446,0,0
SHIPPING,4,0,0,1,0,0,0,0,0,294,0


In [16]:
# Identify misclassified validation samples

error_mask = y_val.to_numpy() != y_val_pred

validation_errors = val_df.loc[
    error_mask,
    ["instruction", "category", "intent"]
].copy()

validation_errors["predicted_category"] = y_val_pred[error_mask]

validation_errors

,instruction,category,intent,predicted_category
166,would you give me information about cancelling...,ORDER,cancel_order,ACCOUNT
1758,correct order370795561790,ORDER,change_order,ACCOUNT
2252,I can't modify theaddress,SHIPPING,change_shipping_address,ACCOUNT
2473,i dont know what i need to do to correct thead...,SHIPPING,change_shipping_address,ACCOUNT
2825,I'd like to correct the bloody dadress can i g...,SHIPPING,change_shipping_address,ACCOUNT
5080,do ya accept card,PAYMENT,check_payment_methods,ACCOUNT
5477,is it possible to check the allowed paymentmod...,PAYMENT,check_payment_methods,DELIVERY
6360,are charges for services reimbursable?,REFUND,check_refund_policy,FEEDBACK
6604,are the service charges refundable?,REFUND,check_refund_policy,CANCEL
6813,are charges for services reimbursable,REFUND,check_refund_policy,FEEDBACK


In [17]:
# Save baseline validation errors

ERRORS_PATH = (
    PROJECT_ROOT / "data" / "processed" /
    "baseline_validation_errors_v1.csv"
)

ERRORS_PATH.parent.mkdir(parents=True, exist_ok=True)

validation_errors.to_csv(
    ERRORS_PATH,
    index=False
)

print("Baseline validation errors saved successfully.")
print("Saved to:", ERRORS_PATH)
print("Total errors:", len(validation_errors))

Baseline validation errors saved successfully.
Saved to: /Users/raghad/Desktop/Lumora/data/processed/baseline_validation_errors_v1.csv
Total errors: 18


In [18]:
from sklearn.metrics import f1_score

baseline_macro_f1 = f1_score(
    y_val,
    y_val_pred,
    average="macro"
)

baseline_weighted_f1 = f1_score(
    y_val,
    y_val_pred,
    average="weighted"
)

print(f"Baseline Macro F1: {baseline_macro_f1:.6f}")
print(f"Baseline Weighted F1: {baseline_weighted_f1:.6f}")

Baseline Macro F1: 0.995880
Baseline Weighted F1: 0.995544


In [19]:
# Save baseline evaluation metrics

baseline_metrics = pd.DataFrame([{
    "model": "Logistic Regression",
    "vectorizer": "TF-IDF",
    "accuracy": val_accuracy,
    "macro_f1": baseline_macro_f1,
    "weighted_f1": baseline_weighted_f1,
    "validation_errors": len(validation_errors),
    "training_samples": len(X_train),
    "validation_samples": len(X_val),
    "tfidf_features": X_train_tfidf.shape[1]
}])

METRICS_PATH = (
    PROJECT_ROOT / "data" / "processed" /
    "baseline_metrics_v1.csv"
)

baseline_metrics.to_csv(METRICS_PATH, index=False)

baseline_metrics

,model,vectorizer,accuracy,macro_f1,weighted_f1,validation_errors,training_samples,validation_samples,tfidf_features
0,Logistic Regression,TF-IDF,0.995546,0.99588,0.995544,18,18798,4041,4876


In [20]:
# Initialize a class-balanced Logistic Regression model

balanced_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

# Train the balanced model

balanced_model.fit(X_train_tfidf, y_train)

print("Balanced Logistic Regression trained successfully.")

Balanced Logistic Regression trained successfully.


In [21]:
# Generate predictions using the balanced model

y_val_pred_balanced = balanced_model.predict(X_val_tfidf)

print("Balanced model predictions generated successfully.")
print("Number of predictions:", len(y_val_pred_balanced))

Balanced model predictions generated successfully.
Number of predictions: 4041


In [22]:
# Calculate balanced model evaluation metrics

balanced_accuracy = accuracy_score(
    y_val,
    y_val_pred_balanced
)

balanced_macro_f1 = f1_score(
    y_val,
    y_val_pred_balanced,
    average="macro"
)

balanced_weighted_f1 = f1_score(
    y_val,
    y_val_pred_balanced,
    average="weighted"
)

print(f"Balanced Accuracy: {balanced_accuracy:.6f}")
print(f"Balanced Macro F1: {balanced_macro_f1:.6f}")
print(f"Balanced Weighted F1: {balanced_weighted_f1:.6f}")

Balanced Accuracy: 0.996041
Balanced Macro F1: 0.996210
Balanced Weighted F1: 0.996038


In [23]:
# Evaluate balanced model performance per category

print(
    classification_report(
        y_val,
        y_val_pred_balanced
    )
)

              precision    recall  f1-score   support

     ACCOUNT       0.99      1.00      1.00       981
      CANCEL       0.99      1.00      1.00       150
     CONTACT       1.00      0.99      0.99       295
    DELIVERY       0.99      1.00      0.99       289
    FEEDBACK       0.99      1.00      1.00       275
     INVOICE       1.00      1.00      1.00       271
       ORDER       1.00      0.99      1.00       573
     PAYMENT       1.00      0.99      1.00       312
      REFUND       1.00      0.99      1.00       449
    SHIPPING       1.00      0.98      0.99       299
SUBSCRIPTION       1.00      1.00      1.00       147

    accuracy                           1.00      4041
   macro avg       1.00      1.00      1.00      4041
weighted avg       1.00      1.00      1.00      4041



In [24]:
# Compare baseline and balanced model errors

baseline_error_mask = y_val.to_numpy() != y_val_pred
balanced_error_mask = y_val.to_numpy() != y_val_pred_balanced

print("Baseline errors:", baseline_error_mask.sum())
print("Balanced errors:", balanced_error_mask.sum())

print(
    "Errors corrected:",
    (baseline_error_mask & ~balanced_error_mask).sum()
)

print(
    "New errors introduced:",
    (~baseline_error_mask & balanced_error_mask).sum()
)

print(
    "Errors shared by both models:",
    (baseline_error_mask & balanced_error_mask).sum()
)

Baseline errors: 18
Balanced errors: 16
Errors corrected: 2
New errors introduced: 0
Errors shared by both models: 16


In [25]:
# Extract remaining errors from the balanced model

balanced_error_mask = y_val.to_numpy() != y_val_pred_balanced

balanced_errors = val_df.loc[
    balanced_error_mask,
    ["instruction", "category", "intent"]
].copy()

balanced_errors["predicted_category"] = (
    y_val_pred_balanced[balanced_error_mask]
)

print("Remaining balanced model errors:", len(balanced_errors))

balanced_errors

Remaining balanced model errors: 16


,instruction,category,intent,predicted_category
166,would you give me information about cancelling...,ORDER,cancel_order,ACCOUNT
1758,correct order370795561790,ORDER,change_order,ACCOUNT
2252,I can't modify theaddress,SHIPPING,change_shipping_address,ACCOUNT
2473,i dont know what i need to do to correct thead...,SHIPPING,change_shipping_address,ACCOUNT
2825,I'd like to correct the bloody dadress can i g...,SHIPPING,change_shipping_address,ACCOUNT
5477,is it possible to check the allowed paymentmod...,PAYMENT,check_payment_methods,DELIVERY
6360,are charges for services reimbursable?,REFUND,check_refund_policy,FEEDBACK
6604,are the service charges refundable?,REFUND,check_refund_policy,CANCEL
6813,are charges for services reimbursable,REFUND,check_refund_policy,FEEDBACK
8440,ucstomer service mail address,CONTACT,contact_customer_service,SHIPPING


In [26]:
# Save balanced model validation errors

BALANCED_ERRORS_PATH = (
    PROJECT_ROOT / "data" / "processed" /
    "balanced_validation_errors_v1.csv"
)

balanced_errors.to_csv(
    BALANCED_ERRORS_PATH,
    index=False
)

print("Balanced validation errors saved successfully.")
print("Saved to:", BALANCED_ERRORS_PATH)
print("Total errors:", len(balanced_errors))

Balanced validation errors saved successfully.
Saved to: /Users/raghad/Desktop/Lumora/data/processed/balanced_validation_errors_v1.csv
Total errors: 16


In [27]:
# Generate detailed classification reports

baseline_report = classification_report(
    y_val,
    y_val_pred,
    output_dict=True,
    zero_division=0
)

balanced_report = classification_report(
    y_val,
    y_val_pred_balanced,
    output_dict=True,
    zero_division=0
)

# Compare F1-score for each category

category_comparison = pd.DataFrame({
    "Baseline_F1": {
        category: baseline_report[category]["f1-score"]
        for category in baseline_model.classes_
    },
    "Balanced_F1": {
        category: balanced_report[category]["f1-score"]
        for category in balanced_model.classes_
    }
})

category_comparison["F1_Improvement"] = (
    category_comparison["Balanced_F1"]
    - category_comparison["Baseline_F1"]
)

category_comparison

,Baseline_F1,Balanced_F1,F1_Improvement
ACCOUNT,0.994425,0.995939,0.001514
CANCEL,0.996678,0.996678,0.000000
CONTACT,0.994889,0.994889,0.000000
DELIVERY,0.994836,0.994836,0.000000
FEEDBACK,0.996377,0.996377,0.000000
INVOICE,1.000000,1.000000,0.000000
ORDER,0.997375,0.997375,0.000000
PAYMENT,0.993548,0.996785,0.003236
REFUND,0.996648,0.995536,-0.001112
SHIPPING,0.989899,0.989899,0.000000


In [28]:
# Configure a new TF-IDF vectorizer with trigrams

tfidf_trigram = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 3),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

print("Trigram TF-IDF vectorizer configured successfully.")

Trigram TF-IDF vectorizer configured successfully.


In [29]:
# Fit the trigram vectorizer on training data only

X_train_tfidf_trigram = tfidf_trigram.fit_transform(X_train)

print("Training Trigram TF-IDF shape:", X_train_tfidf_trigram.shape)
print("Number of features:", X_train_tfidf_trigram.shape[1])
print("Number of documents:", X_train_tfidf_trigram.shape[0])

Training Trigram TF-IDF shape: (18798, 12786)
Number of features: 12786
Number of documents: 18798


In [30]:
# Transform validation data using the fitted trigram vectorizer

X_val_tfidf_trigram = tfidf_trigram.transform(X_val)

print("Validation Trigram TF-IDF shape:", X_val_tfidf_trigram.shape)

Validation Trigram TF-IDF shape: (4041, 12786)


In [31]:
# Train Logistic Regression with Trigram TF-IDF

trigram_balanced_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

trigram_balanced_model.fit(
    X_train_tfidf_trigram,
    y_train
)

print("Trigram Balanced Model trained successfully.")

Trigram Balanced Model trained successfully.


In [32]:
# Generate predictions on validation data

y_val_pred_trigram = trigram_balanced_model.predict(
    X_val_tfidf_trigram
)

print("Trigram validation predictions generated successfully.")
print("Number of predictions:", len(y_val_pred_trigram))

Trigram validation predictions generated successfully.
Number of predictions: 4041


In [33]:
# Evaluate Trigram Balanced Model

trigram_accuracy = accuracy_score(
    y_val,
    y_val_pred_trigram
)

trigram_report = classification_report(
    y_val,
    y_val_pred_trigram,
    output_dict=True
)

trigram_macro_f1 = trigram_report["macro avg"]["f1-score"]
trigram_weighted_f1 = trigram_report["weighted avg"]["f1-score"]

print(f"Trigram Accuracy: {trigram_accuracy:.6f}")
print(f"Trigram Macro F1: {trigram_macro_f1:.6f}")
print(f"Trigram Weighted F1: {trigram_weighted_f1:.6f}")

Trigram Accuracy: 0.995298
Trigram Macro F1: 0.995636
Trigram Weighted F1: 0.995297


In [34]:
# Count Trigram validation errors

trigram_errors = y_val != y_val_pred_trigram

print("Trigram validation errors:", trigram_errors.sum())
print("Trigram validation correct predictions:", (~trigram_errors).sum())

Trigram validation errors: 19
Trigram validation correct predictions: 4022


In [35]:
# Extract Trigram validation errors

trigram_error_df = val_df.loc[
    trigram_errors,
    ["instruction", "category", "intent"]
].copy()

trigram_error_df["predicted_category"] = y_val_pred_trigram[
    trigram_errors.to_numpy()
]

print("Trigram error records:", len(trigram_error_df))
display(trigram_error_df)

Trigram error records: 19


,instruction,category,intent,predicted_category
166,would you give me information about cancelling...,ORDER,cancel_order,ACCOUNT
1758,correct order370795561790,ORDER,change_order,ACCOUNT
2252,I can't modify theaddress,SHIPPING,change_shipping_address,ACCOUNT
2473,i dont know what i need to do to correct thead...,SHIPPING,change_shipping_address,ACCOUNT
2825,I'd like to correct the bloody dadress can i g...,SHIPPING,change_shipping_address,ACCOUNT
5477,is it possible to check the allowed paymentmod...,PAYMENT,check_payment_methods,DELIVERY
6360,are charges for services reimbursable?,REFUND,check_refund_policy,FEEDBACK
6604,are the service charges refundable?,REFUND,check_refund_policy,CANCEL
6813,are charges for services reimbursable,REFUND,check_refund_policy,FEEDBACK
8440,ucstomer service mail address,CONTACT,contact_customer_service,SHIPPING


In [36]:
# Configure Logistic Regression with stronger regularization

tuned_model_c05 = LogisticRegression(
    C=0.5,
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

print("Tuned Logistic Regression (C=0.5) configured successfully.")

Tuned Logistic Regression (C=0.5) configured successfully.


In [37]:
# Train Logistic Regression with C=0.5

tuned_model_c05.fit(
    X_train_tfidf,
    y_train
)

print("Tuned model (C=0.5) trained successfully.")

Tuned model (C=0.5) trained successfully.


In [38]:
# Generate predictions with C=0.5

y_val_pred_c05 = tuned_model_c05.predict(X_val_tfidf)

print("C=0.5 validation predictions generated successfully.")
print("Number of predictions:", len(y_val_pred_c05))

C=0.5 validation predictions generated successfully.
Number of predictions: 4041


In [39]:
 # Evaluate Logistic Regression with C=0.5

c05_accuracy = accuracy_score(y_val, y_val_pred_c05)

c05_report = classification_report(
    y_val,
    y_val_pred_c05,
    output_dict=True
)

c05_macro_f1 = c05_report["macro avg"]["f1-score"]
c05_weighted_f1 = c05_report["weighted avg"]["f1-score"]

print(f"C=0.5 Accuracy: {c05_accuracy:.6f}")
print(f"C=0.5 Macro F1: {c05_macro_f1:.6f}")
print(f"C=0.5 Weighted F1: {c05_weighted_f1:.6f}")

C=0.5 Accuracy: 0.995051
C=0.5 Macro F1: 0.995519
C=0.5 Weighted F1: 0.995050


In [40]:
# Configure Logistic Regression with C=2

tuned_model_c2 = LogisticRegression(
    C=2.0,
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

print("Tuned Logistic Regression (C=2) configured successfully.")

Tuned Logistic Regression (C=2) configured successfully.


In [41]:
# Train Logistic Regression with C=2

tuned_model_c2.fit(
    X_train_tfidf,
    y_train
)

print("Tuned model (C=2) trained successfully.")

Tuned model (C=2) trained successfully.


In [42]:
# Generate predictions with C=2

y_val_pred_c2 = tuned_model_c2.predict(X_val_tfidf)

print("C=2 validation predictions generated successfully.")
print("Number of predictions:", len(y_val_pred_c2))

C=2 validation predictions generated successfully.
Number of predictions: 4041


In [43]:
# Evaluate Logistic Regression with C=2

c2_accuracy = accuracy_score(y_val, y_val_pred_c2)

c2_report = classification_report(
    y_val,
    y_val_pred_c2,
    output_dict=True
)

c2_macro_f1 = c2_report["macro avg"]["f1-score"]
c2_weighted_f1 = c2_report["weighted avg"]["f1-score"]

print(f"C=2 Accuracy: {c2_accuracy:.6f}")
print(f"C=2 Macro F1: {c2_macro_f1:.6f}")
print(f"C=2 Weighted F1: {c2_weighted_f1:.6f}")

C=2 Accuracy: 0.996041
C=2 Macro F1: 0.996061
C=2 Weighted F1: 0.996041


In [44]:
# Count validation errors for C=2

c2_errors = y_val != y_val_pred_c2

print("C=2 validation errors:", c2_errors.sum())
print("C=2 correct predictions:", (~c2_errors).sum())

C=2 validation errors: 16
C=2 correct predictions: 4025


In [45]:
# Compare validation errors between C=1 and C=2

c1_errors = y_val != y_val_pred_balanced
c2_errors = y_val != y_val_pred_c2

print("C=1 errors:", c1_errors.sum())
print("C=2 errors:", c2_errors.sum())

print("Errors corrected by C=2:", (c1_errors & ~c2_errors).sum())
print("New errors introduced by C=2:", (~c1_errors & c2_errors).sum())
print("Errors shared by both models:", (c1_errors & c2_errors).sum())

C=1 errors: 16
C=2 errors: 16
Errors corrected by C=2: 2
New errors introduced by C=2: 2
Errors shared by both models: 14


In [46]:
# Compare per-category F1 scores

c1_report = classification_report(
    y_val,
    y_val_pred_balanced,
    output_dict=True
)

c2_report = classification_report(
    y_val,
    y_val_pred_c2,
    output_dict=True
)

c_comparison = pd.DataFrame({
    "C1_F1": {
        category: c1_report[category]["f1-score"]
        for category in y_val.unique()
    },
    "C2_F1": {
        category: c2_report[category]["f1-score"]
        for category in y_val.unique()
    }
})

c_comparison["Difference"] = (
    c_comparison["C2_F1"] - c_comparison["C1_F1"]
)

display(c_comparison.sort_values("Difference"))

,C1_F1,C2_F1,Difference
CANCEL,0.996678,0.990099,-0.006579
PAYMENT,0.996785,0.995200,-0.001585
ACCOUNT,0.995939,0.995425,-0.000515
ORDER,0.997375,0.997375,0.000000
INVOICE,1.000000,1.000000,0.000000
REFUND,0.995536,0.995536,0.000000
CONTACT,0.994889,0.994889,0.000000
SUBSCRIPTION,1.000000,1.000000,0.000000
SHIPPING,0.989899,0.991597,0.001698
DELIVERY,0.994836,0.996552,0.001715


In [47]:
# Configure Logistic Regression with C=1.5

tuned_model_c15 = LogisticRegression(
    C=1.5,
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

print("Tuned Logistic Regression (C=1.5) configured successfully.")

Tuned Logistic Regression (C=1.5) configured successfully.


In [48]:
# Train Logistic Regression with C=1.5

tuned_model_c15.fit(
    X_train_tfidf,
    y_train
)

print("Tuned model (C=1.5) trained successfully.")

Tuned model (C=1.5) trained successfully.


In [49]:
# Generate predictions with C=1.5

y_val_pred_c15 = tuned_model_c15.predict(X_val_tfidf)

print("C=1.5 validation predictions generated successfully.")
print("Number of predictions:", len(y_val_pred_c15))

C=1.5 validation predictions generated successfully.
Number of predictions: 4041


In [50]:
# Evaluate Logistic Regression with C=1.5

c15_accuracy = accuracy_score(y_val, y_val_pred_c15)

c15_report = classification_report(
    y_val,
    y_val_pred_c15,
    output_dict=True
)

c15_macro_f1 = c15_report["macro avg"]["f1-score"]
c15_weighted_f1 = c15_report["weighted avg"]["f1-score"]

print(f"C=1.5 Accuracy: {c15_accuracy:.6f}")
print(f"C=1.5 Macro F1: {c15_macro_f1:.6f}")
print(f"C=1.5 Weighted F1: {c15_weighted_f1:.6f}")

C=1.5 Accuracy: 0.995793
C=1.5 Macro F1: 0.995861
C=1.5 Weighted F1: 0.995793


In [51]:
# Count validation errors for C=1.5

c15_errors = y_val != y_val_pred_c15

print("C=1.5 validation errors:", c15_errors.sum())
print("C=1.5 correct predictions:", (~c15_errors).sum())

C=1.5 validation errors: 17
C=1.5 correct predictions: 4024


In [52]:
# Compare validation errors between C=1 and C=1.5

c1_errors = y_val != y_val_pred_balanced
c15_errors = y_val != y_val_pred_c15

print("C=1 errors:", c1_errors.sum())
print("C=1.5 errors:", c15_errors.sum())

print("Errors corrected by C=1.5:", (c1_errors & ~c15_errors).sum())
print("New errors introduced by C=1.5:", (~c1_errors & c15_errors).sum())
print("Errors shared by both models:", (c1_errors & c15_errors).sum())

C=1 errors: 16
C=1.5 errors: 17
Errors corrected by C=1.5: 1
New errors introduced by C=1.5: 2
Errors shared by both models: 15


In [53]:
# Configure Logistic Regression with C=0.75

tuned_model_c075 = LogisticRegression(
    C=0.75,
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

print("Tuned Logistic Regression (C=0.75) configured successfully.")

Tuned Logistic Regression (C=0.75) configured successfully.


In [54]:
tuned_model_c075.fit(
    X_train_tfidf,
    y_train
)

print("Tuned model (C=0.75) trained successfully.")

Tuned model (C=0.75) trained successfully.


In [55]:
y_val_pred_c075 = tuned_model_c075.predict(X_val_tfidf)

print("C=0.75 validation predictions completed.")
print("Number of predictions:", len(y_val_pred_c075))

C=0.75 validation predictions completed.
Number of predictions: 4041


In [56]:
print("C=0.75 Accuracy:", accuracy_score(y_val, y_val_pred_c075))

print("\nC=0.75 Classification Report:")
print(classification_report(y_val, y_val_pred_c075, zero_division=0))

C=0.75 Accuracy: 0.9948032665181886

C=0.75 Classification Report:
              precision    recall  f1-score   support

     ACCOUNT       0.99      1.00      0.99       981
      CANCEL       0.99      1.00      1.00       150
     CONTACT       1.00      0.99      0.99       295
    DELIVERY       0.99      1.00      0.99       289
    FEEDBACK       0.99      1.00      1.00       275
     INVOICE       1.00      1.00      1.00       271
       ORDER       1.00      0.99      1.00       573
     PAYMENT       0.99      0.99      0.99       312
      REFUND       1.00      0.99      0.99       449
    SHIPPING       1.00      0.98      0.99       299
SUBSCRIPTION       1.00      1.00      1.00       147

    accuracy                           0.99      4041
   macro avg       1.00      1.00      1.00      4041
weighted avg       0.99      0.99      0.99      4041



In [57]:
errors_c075 = (y_val != y_val_pred_c075).sum()
correct_c075 = (y_val == y_val_pred_c075).sum()

print("C=0.75 validation errors:", errors_c075)
print("C=0.75 correct predictions:", correct_c075)

C=0.75 validation errors: 21
C=0.75 correct predictions: 4020


In [58]:
errors_c1 = set(np.where(y_val.to_numpy() != balanced_model.predict(X_val_tfidf))[0])
errors_c075 = set(np.where(y_val.to_numpy() != y_val_pred_c075)[0])

print("C=1 errors:", len(errors_c1))
print("C=0.75 errors:", len(errors_c075))

print("Errors corrected by C=0.75:", len(errors_c1 - errors_c075))
print("New errors introduced by C=0.75:", len(errors_c075 - errors_c1))
print("Errors shared by both models:", len(errors_c1 & errors_c075))

C=1 errors: 16
C=0.75 errors: 21
Errors corrected by C=0.75: 0
New errors introduced by C=0.75: 5
Errors shared by both models: 16


In [59]:
# Extract validation errors from the selected C=1 model

error_mask = y_val.to_numpy() != y_val_pred_balanced

error_df = val_df.loc[error_mask].copy()

error_df["true_category"] = y_val.to_numpy()[error_mask]
error_df["predicted_category"] = y_val_pred_balanced[error_mask]

error_df = error_df[
    [
        "instruction",
        "true_category",
        "predicted_category",
        "intent"
    ]
].reset_index(drop=True)

print("Total validation errors:", len(error_df))

display(error_df)

Total validation errors: 16


,instruction,true_category,predicted_category,intent
0,would you give me information about cancelling...,ORDER,ACCOUNT,cancel_order
1,correct order370795561790,ORDER,ACCOUNT,change_order
2,I can't modify theaddress,SHIPPING,ACCOUNT,change_shipping_address
3,i dont know what i need to do to correct thead...,SHIPPING,ACCOUNT,change_shipping_address
4,I'd like to correct the bloody dadress can i g...,SHIPPING,ACCOUNT,change_shipping_address
5,is it possible to check the allowed paymentmod...,PAYMENT,DELIVERY,check_payment_methods
6,are charges for services reimbursable?,REFUND,FEEDBACK,check_refund_policy
7,are the service charges refundable?,REFUND,CANCEL,check_refund_policy
8,are charges for services reimbursable,REFUND,FEEDBACK,check_refund_policy
9,ucstomer service mail address,CONTACT,SHIPPING,contact_customer_service


In [60]:
# Inspect prediction confidence for validation errors

error_probabilities = balanced_model.predict_proba(
    tfidf.transform(error_df["instruction"])
)

classes = balanced_model.classes_

error_df["predicted_probability"] = error_probabilities.max(axis=1)

error_df["second_best_category"] = [
    classes[np.argsort(row)[-2]]
    for row in error_probabilities
]

error_df["second_best_probability"] = [
    np.sort(row)[-2]
    for row in error_probabilities
]

error_df["confidence_gap"] = (
    error_df["predicted_probability"]
    - error_df["second_best_probability"]
)

display(
    error_df[
        [
            "instruction",
            "true_category",
            "predicted_category",
            "predicted_probability",
            "second_best_category",
            "second_best_probability",
            "confidence_gap"
        ]
    ].round(4)
)

,instruction,true_category,predicted_category,predicted_probability,second_best_category,second_best_probability,confidence_gap
0,would you give me information about cancelling...,ORDER,ACCOUNT,0.3162,SHIPPING,0.1294,0.1868
1,correct order370795561790,ORDER,ACCOUNT,0.2235,ORDER,0.1489,0.0746
2,I can't modify theaddress,SHIPPING,ACCOUNT,0.1893,SHIPPING,0.1732,0.0161
3,i dont know what i need to do to correct thead...,SHIPPING,ACCOUNT,0.1898,REFUND,0.1400,0.0499
4,I'd like to correct the bloody dadress can i g...,SHIPPING,ACCOUNT,0.1859,ORDER,0.1613,0.0247
5,is it possible to check the allowed paymentmod...,PAYMENT,DELIVERY,0.2497,CANCEL,0.2320,0.0177
6,are charges for services reimbursable?,REFUND,FEEDBACK,0.2715,CANCEL,0.2651,0.0064
7,are the service charges refundable?,REFUND,CANCEL,0.4071,REFUND,0.1036,0.3035
8,are charges for services reimbursable,REFUND,FEEDBACK,0.2715,CANCEL,0.2651,0.0064
9,ucstomer service mail address,CONTACT,SHIPPING,0.5353,CONTACT,0.1362,0.3991


In [ ]:
# Inspect similar training examples for the first error

search_text = "correct order"

similar_examples = train_df[
    train_df["instruction"].str.contains(
        search_text,
        case=False,
        na=False
    )
][["instruction", "category", "intent"]]

display(similar_examples.head(20))

print("Matching training examples:", len(similar_examples))

In [61]:
# Inspect similar training examples for the first error

search_text = "correct order"

similar_examples = train_df[
    train_df["instruction"].str.contains(
        search_text,
        case=False,
        na=False
    )
][["instruction", "category", "intent"]]

display(similar_examples.head(20))

print("Matching training examples:", len(similar_examples))

,instruction,category,intent
1058,correct order {{Order Number}},ORDER,change_order
1084,"I want to correct order {{Order Number}}, will...",ORDER,change_order
1085,correct order {{Order Number}},ORDER,change_order
1124,correct order {{Order Number}},ORDER,change_order
1141,"I have to correct order {{Order Number}}, coul...",ORDER,change_order
1240,how do I correct order {{Order Number}}?,ORDER,change_order
1294,correct order {{Order Number}},ORDER,change_order
1330,help to correct order {{Order Number}},ORDER,change_order
1404,I would like to correct order {{Order Number}}...,ORDER,change_order
1406,how to correct order {{Order Number}}?,ORDER,change_order


Matching training examples: 22


In [62]:
similar_examples = train_df[
    train_df["instruction"].str.contains(
        "cancel",
        case=False,
        na=False
    )
][["instruction", "category", "intent"]]

display(similar_examples.head(20))

print("Matching training examples:", len(similar_examples))

,instruction,category,intent
1,i have a question about cancelling oorder {{Or...,ORDER,cancel_order
2,i need help cancelling puchase {{Order Number}},ORDER,cancel_order
3,I need to cancel purchase {{Order Number}},ORDER,cancel_order
4,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order
5,can you help me cancel order {{Order Number}}?,ORDER,cancel_order
7,I am trying to cancel purchase {{Order Number}},ORDER,cancel_order
8,I have got to cancel purchase {{Order Number}},ORDER,cancel_order
9,i need help canceling purchase {{Order Number}},ORDER,cancel_order
10,i dont know what to do to cancel order {{Order...,ORDER,cancel_order
11,I have a problem with cancelling purchase {{Or...,ORDER,cancel_order


Matching training examples: 1101


In [63]:
similar_examples = train_df[
    train_df["instruction"].str.contains(
        "shipping address",
        case=False,
        na=False
    )
][["instruction", "category", "intent"]]

display(similar_examples.head(20))

print("Matching training examples:", len(similar_examples))

,instruction,category,intent
1999,help modifying the shipping address,SHIPPING,change_shipping_address
2010,"my shipping address changed, can I update it?",SHIPPING,change_shipping_address
2013,I need help trying to change my shipping address,SHIPPING,change_shipping_address
2015,there are problems trying to modify the shippi...,SHIPPING,change_shipping_address
2021,i have issues trying to modify my shipping add...,SHIPPING,change_shipping_address
2022,"I chose a wrong shipping address, help me edit...",SHIPPING,change_shipping_address
2029,could you help me to correct my shipping address?,SHIPPING,change_shipping_address
2030,help me updating my shipping address,SHIPPING,change_shipping_address
2031,"the shipping address changed, how can I correc...",SHIPPING,change_shipping_address
2034,can you tell me more about modifying my shippi...,SHIPPING,change_shipping_address


Matching training examples: 538


In [64]:
similar_examples = train_df[
    train_df["instruction"].str.contains(
        "payment method",
        case=False,
        na=False
    )
][["instruction", "category", "intent"]]

display(similar_examples.head(20))

print("Matching training examples:", len(similar_examples))

,instruction,category,intent
4919,I nneed help to list the accepted payment methods,PAYMENT,check_payment_methods
4921,see ur payment methods,PAYMENT,check_payment_methods
4926,I have to see what payment methods you acecpt,PAYMENT,check_payment_methods
4929,where to see your accepted payment methods?,PAYMENT,check_payment_methods
4931,"I have to list your accepted payment methods, ...",PAYMENT,check_payment_methods
4934,i want help to check ur accepted payment methods,PAYMENT,check_payment_methods
4942,"I have to check your payment methods, how do I...",PAYMENT,check_payment_methods
4943,is it possible to see your available payment m...,PAYMENT,check_payment_methods
4947,need help to check ur payment methods,PAYMENT,check_payment_methods
4950,I am trying to see your accepted payment methods,PAYMENT,check_payment_methods


Matching training examples: 218


In [65]:
similar_examples = train_df[
    train_df["instruction"].str.contains(
        "refund",
        case=False,
        na=False
    )
][["instruction", "category", "intent"]]

display(similar_examples.head(20))

print("Matching training examples:", len(similar_examples))

,instruction,category,intent
5923,can I see in which cases can I ask to be refun...,REFUND,check_refund_policy
5924,"I have to see your refund policy, can you help...",REFUND,check_refund_policy
5930,can you help me check in which cases can I ask...,REFUND,check_refund_policy
5933,wanna check ur refund policy i need help,REFUND,check_refund_policy
5934,is it possible to see in what cases can I requ...,REFUND,check_refund_policy
5938,I want assistance checking how long refunds take,REFUND,check_refund_policy
5940,want assistance to check how long refunds take,REFUND,check_refund_policy
5941,help me seeing in which situations can I ask f...,REFUND,check_refund_policy
5942,can you help me seeing in what cases can I req...,REFUND,check_refund_policy
5950,is it possibpe to check how long refunds take,REFUND,check_refund_policy


Matching training examples: 946


In [66]:
similar_examples = train_df[
    train_df["instruction"].str.contains(
        r"reimburs|service charges|refundable",
        case=False,
        na=False,
        regex=True
    )
][["instruction", "category", "intent"]]

display(similar_examples.head(30))

print("Matching training examples:", len(similar_examples))

,instruction,category,intent
5920,can I check in what cases can I ask to be reim...,REFUND,check_refund_policy
5922,I try to check in which cases can I ask to be ...,REFUND,check_refund_policy
5925,how can I check your reimbursement policy?,REFUND,check_refund_policy
5926,help me see in what cases can I ask to be reim...,REFUND,check_refund_policy
5935,help seeing in what situations can I request r...,REFUND,check_refund_policy
5937,assistance seeing in what cases can i request ...,REFUND,check_refund_policy
5947,help to see in what cases can I ask to be reim...,REFUND,check_refund_policy
5951,help checking in what cases can I request to b...,REFUND,check_refund_policy
5953,can ya show me in what situations can i ask to...,REFUND,check_refund_policy
5957,help seeing in which situations can I ask to b...,REFUND,check_refund_policy


Matching training examples: 488


In [67]:
similar_examples = train_df[
    train_df["instruction"].str.contains(
        r"contact|customer service|human agent",
        case=False,
        na=False,
        regex=True
    )
][["instruction", "category", "intent"]]

display(similar_examples.head(30))

print("Matching training examples:", len(similar_examples))

,instruction,category,intent
7922,I need to see what hours I can contact custome...,CONTACT,contact_customer_service
7929,i do not know what to do to speak to customer ...,CONTACT,contact_customer_service
7932,help me check what hours I can contact custome...,CONTACT,contact_customer_service
7933,"I want to contact customer support, could you ...",CONTACT,contact_customer_service
7935,can I check at what time I can reach customer ...,CONTACT,contact_customer_service
7936,i haqve to see at what time i can contact cust...,CONTACT,contact_customer_service
7939,can i check at what time customer service avai...,CONTACT,contact_customer_service
7941,help me check what hours i can contact custome...,CONTACT,contact_customer_service
7945,i dont know how to speak with customer service,CONTACT,contact_customer_service
7948,I do not know how to call customer service,CONTACT,contact_customer_service


Matching training examples: 494


In [68]:
human_agent_examples = train_df[
    train_df["instruction"].str.contains(
        r"human agent|speak to someone|contact someone|talk to someone",
        case=False,
        na=False,
        regex=True
    )
][["instruction", "category", "intent"]]

display(human_agent_examples.head(30))

print("Matching training examples:", len(human_agent_examples))

,instruction,category,intent
8917,how do i contact someone,CONTACT,contact_human_agent
8919,can i contact someone,CONTACT,contact_human_agent
8929,need help to contact a human agent,CONTACT,contact_human_agent
8932,I want assistance to talk to a human agent,CONTACT,contact_human_agent
8940,"I can't understand a word you say, speak with ...",CONTACT,contact_human_agent
8941,help me contacting a fucking human agent,CONTACT,contact_human_agent
8950,need assistance to talk to someone,CONTACT,contact_human_agent
8971,help to speak with a human agent,CONTACT,contact_human_agent
8973,want assitsance to talk with a human agent,CONTACT,contact_human_agent
8979,i do not know how i can talk to a human agent,CONTACT,contact_human_agent


Matching training examples: 111


In [69]:
similar_examples = train_df[
    train_df["instruction"].str.contains(
        r"payment issue|payment problem|problem with payment",
        case=False,
        na=False,
        regex=True
    )
][["instruction", "category", "intent"]]

display(similar_examples.head(30))

print("Matching training examples:", len(similar_examples))

,instruction,category,intent
17897,I do not know how I can notify of a payment issue,PAYMENT,payment_issue
17898,can uhelpme informing of a problem with payments,PAYMENT,payment_issue
17901,I try to inform of a payment issue,PAYMENT,payment_issue
17917,assistance to inform of a damn payment issue,PAYMENT,payment_issue
17936,I want assistance informing of a payment issue,PAYMENT,payment_issue
17967,how do I notify of payment problems?,PAYMENT,payment_issue
17975,notify of fucking payment issues,PAYMENT,payment_issue
17983,how to report a problem with payment,PAYMENT,payment_issue
17985,what do I need to do to notify of a problem wi...,PAYMENT,payment_issue
17987,I don't know how I could notify of payment issues,PAYMENT,payment_issue


Matching training examples: 105


In [70]:
similar_examples = train_df[
    train_df["instruction"].str.contains(
        r"shipping address|delivery address",
        case=False,
        na=False,
        regex=True
    )
][["instruction", "category", "intent"]]

display(similar_examples.head(30))

print("Matching training examples:", len(similar_examples))

,instruction,category,intent
1999,help modifying the shipping address,SHIPPING,change_shipping_address
2002,I want to modify my delivery address,SHIPPING,change_shipping_address
2008,i need assistance changing the delivery address,SHIPPING,change_shipping_address
2010,"my shipping address changed, can I update it?",SHIPPING,change_shipping_address
2012,i want support trying to change my delivery ad...,SHIPPING,change_shipping_address
2013,I need help trying to change my shipping address,SHIPPING,change_shipping_address
2015,there are problems trying to modify the shippi...,SHIPPING,change_shipping_address
2021,i have issues trying to modify my shipping add...,SHIPPING,change_shipping_address
2022,"I chose a wrong shipping address, help me edit...",SHIPPING,change_shipping_address
2027,help me to change the delivery address,SHIPPING,change_shipping_address


Matching training examples: 1058


In [71]:
similar_examples = train_df[
    train_df["instruction"].str.contains(
        r"set up.*address|setup.*address|add.*address|new.*address",
        case=False,
        na=False,
        regex=True
    )
][["instruction", "category", "intent"]]

display(similar_examples.head(30))

print("Matching training examples:", len(similar_examples))

,instruction,category,intent
22883,I want help to sdt up the new shipping address,SHIPPING,set_up_shipping_address
22885,I want help to set up my delivery address,SHIPPING,set_up_shipping_address
22888,what do I have to do to enter my new shipping ...,SHIPPING,set_up_shipping_address
22891,i need help to set up a delivery address,SHIPPING,set_up_shipping_address
22894,need help to set up my delivery address,SHIPPING,set_up_shipping_address
22898,i want help setting up a new shipping address,SHIPPING,set_up_shipping_address
22900,I have an issue setting a new delivery address up,SHIPPING,set_up_shipping_address
22910,"I have to submit my new delivery address, will...",SHIPPING,set_up_shipping_address
22921,i want aswistance to set up the new delivery a...,SHIPPING,set_up_shipping_address
22923,"I have to enter a new shipping address, how do...",SHIPPING,set_up_shipping_address


Matching training examples: 251


In [72]:
similar_examples = train_df[
    train_df["instruction"].str.contains(
        r"track.*order|order.*track|tracking.*order|order status|ETA",
        case=False,
        na=False,
        regex=True
    )
][["instruction", "category", "intent"]]

display(similar_examples.head(30))

print("Matching training examples:", len(similar_examples))

,instruction,category,intent
13913,changing details on gold account,ACCOUNT,edit_account
13920,changing details on {{Account Type}} account,ACCOUNT,edit_account
13931,modifying details on {{Account Category}} account,ACCOUNT,edit_account
13941,update details on {{Account Type}} account,ACCOUNT,edit_account
13948,how do I edit some of the details included on ...,ACCOUNT,edit_account
13951,edit details on {{Account Type}} account,ACCOUNT,edit_account
13959,need assistance changing the details on my pro...,ACCOUNT,edit_account
13960,update details on {{Account Category}} account,ACCOUNT,edit_account
13965,updating details on platinum account,ACCOUNT,edit_account
13973,modifying details on pro account,ACCOUNT,edit_account


Matching training examples: 426


In [73]:
similar_examples = train_df[
    train_df["intent"].eq("track_order")
][["instruction", "category", "intent"]]

display(similar_examples.head(30))

print("Matching training examples:", len(similar_examples))

,instruction,category,intent
24880,ETA of purchase {{Order Number}},ORDER,track_order
24881,need to check the eta of order {{Order Number}},ORDER,track_order
24882,I cannot see purchase {{Order Number}} current...,ORDER,track_order
24883,can uhelp me to check the fucking eta of the o...,ORDER,track_order
24884,seeing status of purchase {{Order Number}},ORDER,track_order
24885,order {{Order Number}} status,ORDER,track_order
24886,I have to check purchase {{Order Number}} stat...,ORDER,track_order
24887,I need to see purchase {{Order Number}} curren...,ORDER,track_order
24888,I need assistance seeing purchase {{Order Numb...,ORDER,track_order
24890,I am trying to check order {{Order Number}} cu...,ORDER,track_order


Matching training examples: 685


In [74]:
import json
from pathlib import Path

model_config = {
    "model_name": "Balanced TF-IDF Logistic Regression",
    "status": "selected_candidate_pending_test_evaluation",

    "vectorizer": {
        "type": "TfidfVectorizer",
        "lowercase": True,
        "ngram_range": [1, 2],
        "min_df": 2,
        "max_df": 0.95,
        "sublinear_tf": True
    },

    "classifier": {
        "type": "LogisticRegression",
        "C": 1.0,
        "class_weight": "balanced",
        "max_iter": 1000,
        "random_state": 42
    },

    "validation_results": {
        "accuracy": 0.996041,
        "macro_f1": 0.996210,
        "weighted_f1": 0.996038,
        "errors": 16,
        "validation_records": 4041
    },

    "selection_note": (
        "Selected based on validation performance and comparison "
        "with alternative configurations. Test set remains untouched."
    )
}

output_path = (
    Path.home()
    / "Desktop"
    / "Lumora"
    / "data"
    / "processed"
    / "selected_model_config_v1.json"
)

output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as file:
    json.dump(model_config, file, indent=4)

print("Model configuration saved successfully.")
print(f"Location: {output_path}")

Model configuration saved successfully.
Location: /Users/raghad/Desktop/Lumora/data/processed/selected_model_config_v1.json


In [75]:
print("Vectorizer:")
print(tfidf)

print("\nClassifier:")
print(balanced_model)

print("\nValidation predictions:", len(y_val_pred_balanced))
print("Test records:", len(X_test))

print("\nSelected C:", balanced_model.C)
print("Class weight:", balanced_model.class_weight)
print("N-gram range:", tfidf.ngram_range)

Vectorizer:
TfidfVectorizer(max_df=0.95, min_df=2, ngram_range=(1, 2), sublinear_tf=True)

Classifier:
LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

Validation predictions: 4041
Test records: 4033

Selected C: 1.0
Class weight: balanced
N-gram range: (1, 2)


In [76]:
# Transform Test data using the already-fitted TF-IDF vectorizer
X_test_tfidf = tfidf.transform(X_test)

# Generate predictions
y_test_pred = balanced_model.predict(X_test_tfidf)

# Calculate evaluation metrics
test_accuracy = accuracy_score(y_test, y_test_pred)

test_report = classification_report(
    y_test,
    y_test_pred,
    output_dict=True,
    zero_division=0
)

test_macro_f1 = test_report["macro avg"]["f1-score"]
test_weighted_f1 = test_report["weighted avg"]["f1-score"]

# Display results
print("FINAL TEST RESULTS")
print("=" * 40)

print(f"Accuracy:    {test_accuracy:.6f}")
print(f"Macro F1:    {test_macro_f1:.6f}")
print(f"Weighted F1: {test_weighted_f1:.6f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_test_pred,
        zero_division=0
    )
)

print("Total Test Errors:", (y_test.to_numpy() != y_test_pred).sum())

FINAL TEST RESULTS
Accuracy:    0.997768
Macro F1:    0.997487
Weighted F1: 0.997771

Classification Report:
              precision    recall  f1-score   support

     ACCOUNT       1.00      1.00      1.00       899
      CANCEL       0.98      1.00      0.99       129
     CONTACT       1.00      1.00      1.00       293
    DELIVERY       1.00      1.00      1.00       289
    FEEDBACK       1.00      1.00      1.00       309
     INVOICE       1.00      1.00      1.00       303
       ORDER       1.00      0.99      1.00       620
     PAYMENT       1.00      0.99      1.00       290
      REFUND       1.00      1.00      1.00       475
    SHIPPING       0.99      1.00      0.99       287
SUBSCRIPTION       1.00      1.00      1.00       139

    accuracy                           1.00      4033
   macro avg       1.00      1.00      1.00      4033
weighted avg       1.00      1.00      1.00      4033

Total Test Errors: 9


In [77]:
test_error_mask = y_test.to_numpy() != y_test_pred

test_error_df = test_df.loc[test_error_mask].copy()

test_error_df["true_category"] = y_test.to_numpy()[test_error_mask]
test_error_df["predicted_category"] = y_test_pred[test_error_mask]

test_error_df = test_error_df[
    ["instruction", "true_category", "predicted_category", "intent"]
].reset_index(drop=True)

display(test_error_df)

print("Total Test Errors:", len(test_error_df))

,instruction,true_category,predicted_category,intent
0,update purchase113542617735902,ORDER,SHIPPING,change_order
1,can I update order00004587345?,ORDER,SHIPPING,change_order
2,modify order00123842,ORDER,SHIPPING,change_order
3,modify order00004587345,ORDER,SHIPPING,change_order
4,i have a trouble updating my addrses,SHIPPING,ACCOUNT,change_shipping_address
5,can uhelp me check the bloody allowed paymentm...,PAYMENT,CANCEL,check_payment_methods
6,are service charges reimbursable?,REFUND,CANCEL,check_refund_policy
7,where do i report an issue with paynents,PAYMENT,ACCOUNT,payment_issue
8,want assistance to make a purchse,ORDER,FEEDBACK,place_order


Total Test Errors: 9


In [78]:
import joblib

print("Vectorizer fitted:", hasattr(tfidf, "vocabulary_"))
print("Classifier fitted:", hasattr(balanced_model, "classes_"))

print("Number of TF-IDF features:", len(tfidf.vocabulary_))
print("Number of categories:", len(balanced_model.classes_))
print("Categories:", balanced_model.classes_)

Vectorizer fitted: True
Classifier fitted: True
Number of TF-IDF features: 4876
Number of categories: 11
Categories: ['ACCOUNT' 'CANCEL' 'CONTACT' 'DELIVERY' 'FEEDBACK' 'INVOICE' 'ORDER'
 'PAYMENT' 'REFUND' 'SHIPPING' 'SUBSCRIPTION']


In [79]:
import joblib
from pathlib import Path

PROJECT_ROOT = Path.home() / "Desktop" / "Lumora"

MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "lumora_model_v1.joblib"
)

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

model_artifact = {
    "vectorizer": tfidf,
    "classifier": balanced_model,
    "categories": balanced_model.classes_.tolist(),
    "model_name": "Balanced TF-IDF Logistic Regression",
    "version": "1.0"
}

joblib.dump(model_artifact, MODEL_PATH)

print("Model saved successfully.")
print(f"Location: {MODEL_PATH}")
print(f"File size: {MODEL_PATH.stat().st_size / 1024:.2f} KB")

Model saved successfully.
Location: /Users/raghad/Desktop/Lumora/models/lumora_model_v1.joblib
File size: 538.25 KB


In [80]:
import joblib

loaded_artifact = joblib.load(MODEL_PATH)

loaded_tfidf = loaded_artifact["vectorizer"]
loaded_model = loaded_artifact["classifier"]

print("Model loaded successfully.")
print("Version:", loaded_artifact["version"])
print("Categories:", loaded_artifact["categories"])

test_text = [
    "I want to track my order",
    "I need help with a refund",
    "I have a problem with my payment"
]

text_vectors = loaded_tfidf.transform(test_text)
predictions = loaded_model.predict(text_vectors)

for text, prediction in zip(test_text, predictions):
    print(f"\nInput: {text}")
    print(f"Predicted category: {prediction}")

Model loaded successfully.
Version: 1.0
Categories: ['ACCOUNT', 'CANCEL', 'CONTACT', 'DELIVERY', 'FEEDBACK', 'INVOICE', 'ORDER', 'PAYMENT', 'REFUND', 'SHIPPING', 'SUBSCRIPTION']

Input: I want to track my order
Predicted category: REFUND

Input: I need help with a refund
Predicted category: REFUND

Input: I have a problem with my payment
Predicted category: PAYMENT


In [81]:
import pandas as pd

sample_probabilities = loaded_model.predict_proba(text_vectors)

for text, probabilities in zip(test_text, sample_probabilities):

    results = pd.DataFrame({
        "category": loaded_model.classes_,
        "probability": probabilities
    })

    results = results.sort_values(
        "probability",
        ascending=False
    ).head(5)

    print(f"\nInput: {text}")
    display(results.reset_index(drop=True))


Input: I want to track my order


,category,probability
0,REFUND,0.388478
1,ORDER,0.255841
2,ACCOUNT,0.079245
3,DELIVERY,0.070433
4,SHIPPING,0.039986



Input: I need help with a refund


,category,probability
0,REFUND,0.625393
1,CONTACT,0.102674
2,PAYMENT,0.074555
3,ACCOUNT,0.059612
4,ORDER,0.040558



Input: I have a problem with my payment


,category,probability
0,PAYMENT,0.720985
1,ACCOUNT,0.184212
2,SHIPPING,0.028072
3,CONTACT,0.012824
4,REFUND,0.010293


In [82]:
track_examples = train_df[
    train_df["intent"].eq("track_order")
][["instruction", "category", "intent"]]

display(
    track_examples[
        track_examples["instruction"].str.contains(
            r"track|tracking|status|ETA",
            case=False,
            na=False,
            regex=True
        )
    ].head(40)
)

,instruction,category,intent
24880,ETA of purchase {{Order Number}},ORDER,track_order
24881,need to check the eta of order {{Order Number}},ORDER,track_order
24882,I cannot see purchase {{Order Number}} current...,ORDER,track_order
24883,can uhelp me to check the fucking eta of the o...,ORDER,track_order
24884,seeing status of purchase {{Order Number}},ORDER,track_order
24885,order {{Order Number}} status,ORDER,track_order
24886,I have to check purchase {{Order Number}} stat...,ORDER,track_order
24887,I need to see purchase {{Order Number}} curren...,ORDER,track_order
24888,I need assistance seeing purchase {{Order Numb...,ORDER,track_order
24890,I am trying to check order {{Order Number}} cu...,ORDER,track_order


In [83]:
from pathlib import Path

import joblib
import pandas as pd


PROJECT_ROOT = Path(__file__).resolve().parent.parent

MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "lumora_model_v1.joblib"
)

NameError: name '__file__' is not defined

In [84]:
from pathlib import Path

import joblib
import pandas as pd


PROJECT_ROOT = Path(__file__).resolve().parent.parent

MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "lumora_model_v1.joblib"
)

NameError: name '__file__' is not defined

In [87]:
def load_model():
    """
    Load the trained Lumora model artifact.
    """
    if not MODEL_PATH.exists():
        raise FileNotFoundError(
            f"Model file not found: {MODEL_PATH}"
        )

    artifact = joblib.load(MODEL_PATH)

    return artifact

In [88]:
if __name__ == "__main__":
    artifact = load_model()
    print("Model loaded successfully!")
    print("Categories:", artifact["categories"])

Model loaded successfully!
Categories: ['ACCOUNT', 'CANCEL', 'CONTACT', 'DELIVERY', 'FEEDBACK', 'INVOICE', 'ORDER', 'PAYMENT', 'REFUND', 'SHIPPING', 'SUBSCRIPTION']


In [91]:
def predict_category(text):
    """
    Predict the category of a customer support ticket.
    """
    artifact = load_model()

    vectorizer = artifact["vectorizer"]
    classifier = artifact["classifier"]

    text_vector = vectorizer.transform([text])

    prediction = classifier.predict(text_vector)[0]

    return prediction

In [92]:
if __name__ == "__main__":
    artifact = load_model()
    print("Model loaded successfully!")
    print("Categories:", artifact["categories"])

    result = predict_category("I want to track my order")
    print("Predicted category:", result)

Model loaded successfully!
Categories: ['ACCOUNT', 'CANCEL', 'CONTACT', 'DELIVERY', 'FEEDBACK', 'INVOICE', 'ORDER', 'PAYMENT', 'REFUND', 'SHIPPING', 'SUBSCRIPTION']
Predicted category: REFUND


In [93]:
import joblib
import pandas as pd

# Load the saved model
artifact = joblib.load(
    PROJECT_ROOT / "models" / "lumora_model_v1.joblib"
)

vectorizer = artifact["vectorizer"]
classifier = artifact["classifier"]

# New manually written examples
challenge_cases = [
    ("Could you tell me where my package is?", "ORDER"),
    ("I want to track my order", "ORDER"),

    ("Please change the address for my delivery", "SHIPPING"),
    ("The shipping address needs to be corrected", "SHIPPING"),

    ("My card payment was declined", "PAYMENT"),
    ("What payment options do you accept?", "PAYMENT"),

    ("When will I receive my money back?", "REFUND"),
    ("Can I get a refund for this purchase?", "REFUND"),

    ("I need to speak with a real person", "CONTACT"),
    ("How can I reach your support team?", "CONTACT"),

    ("I cannot log into my account", "ACCOUNT"),
    ("How do I reset my password?", "ACCOUNT"),

    ("I want to cancel my purchase", "CANCEL"),
    ("How can I cancel this order?", "CANCEL"),

    ("Please send me a copy of my invoice", "INVOICE"),
    ("Where can I download my bill?", "INVOICE"),

    ("My package has not arrived yet", "DELIVERY"),
    ("When should my delivery arrive?", "DELIVERY"),

    ("The product arrived damaged", "FEEDBACK"),
    ("I would like to share my experience", "FEEDBACK"),

    ("How do I subscribe to your newsletter?", "SUBSCRIPTION"),
    ("I want to stop receiving your newsletter", "SUBSCRIPTION"),
]

challenge_df = pd.DataFrame(
    challenge_cases,
    columns=["instruction", "expected_category"]
)

# Generate predictions
challenge_vectors = vectorizer.transform(
    challenge_df["instruction"]
)

challenge_df["predicted_category"] = classifier.predict(
    challenge_vectors
)

challenge_df["correct"] = (
    challenge_df["expected_category"]
    == challenge_df["predicted_category"]
)

# Display results
print("Generalization Audit Results")
print("=" * 45)

print("Total examples:", len(challenge_df))
print("Correct predictions:", challenge_df["correct"].sum())
print("Incorrect predictions:", (~challenge_df["correct"]).sum())

print(
    "Accuracy:",
    round(challenge_df["correct"].mean() * 100, 2),
    "%"
)

print("\nIncorrect Predictions:")
display(
    challenge_df.loc[
        ~challenge_df["correct"]
    ]
)

Generalization Audit Results
Total examples: 22
Correct predictions: 15
Incorrect predictions: 7
Accuracy: 68.18 %

Incorrect Predictions:


,instruction,expected_category,predicted_category,correct
0,Could you tell me where my package is?,ORDER,DELIVERY,False
1,I want to track my order,ORDER,REFUND,False
12,I want to cancel my purchase,CANCEL,ORDER,False
13,How can I cancel this order?,CANCEL,ORDER,False
16,My package has not arrived yet,DELIVERY,ACCOUNT,False
18,The product arrived damaged,FEEDBACK,ORDER,False
19,I would like to share my experience,FEEDBACK,ACCOUNT,False


In [94]:
# Review all challenge predictions by category

category_review = (
    challenge_df
    .groupby("expected_category")
    .agg(
        total_examples=("correct", "count"),
        correct_predictions=("correct", "sum"),
    )
)

category_review["incorrect_predictions"] = (
    category_review["total_examples"]
    - category_review["correct_predictions"]
)

category_review["accuracy"] = (
    category_review["correct_predictions"]
    / category_review["total_examples"]
    * 100
).round(2)

display(category_review)

,total_examples,correct_predictions,incorrect_predictions,accuracy
expected_category,,,,
ACCOUNT,2,2,0,100.0
CANCEL,2,0,2,0.0
CONTACT,2,2,0,100.0
DELIVERY,2,1,1,50.0
FEEDBACK,2,0,2,0.0
INVOICE,2,2,0,100.0
ORDER,2,0,2,0.0
PAYMENT,2,2,0,100.0
REFUND,2,2,0,100.0


In [95]:
categories_to_review = [
    "ORDER",
    "CANCEL",
    "FEEDBACK",
    "DELIVERY"
]

for category in categories_to_review:

    print("\n" + "=" * 65)
    print(f"CATEGORY: {category}")
    print("=" * 65)

    examples = (
        train_df.loc[
            train_df["category"] == category,
            ["instruction", "intent"]
        ]
        .sample(
            n=min(
                10,
                (train_df["category"] == category).sum()
            ),
            random_state=42
        )
    )

    display(examples)


CATEGORY: ORDER


,instruction,intent
1465,remove item from order {{Order Number}},change_order
25314,I try to see the current status of purchase {{...,track_order
651,help me canceling purchase {{Order Number}},cancel_order
25575,can uhelp me see the currnt status of order {{...,track_order
637,i do not know how i can canmcel order {{Order ...,cancel_order
1961,I want to add something to purchase {{Order Nu...,change_order
19612,how to purchase several articles?,place_order
19505,"I need to earn an article, help me",place_order
1514,help me modifying purchase {{Order Number}},change_order
1836,remove something from purchase {{Order Number}},change_order



CATEGORY: CANCEL


,instruction,intent
3477,want help ot check the withdrawal charge,check_cancellation_fee
3190,i cannot find anything about the early termina...,check_cancellation_fee
3642,where doi see the withdrawal penalties,check_cancellation_fee
3875,"I want to check the termination charge, help me",check_cancellation_fee
3361,"I cannot find the early termination fee, I nee...",check_cancellation_fee
3478,i want assistance seeing the cancellation charge,check_cancellation_fee
3413,Ineed assistance to see the withdrawal charges,check_cancellation_fee
3241,where can i see the termniation penalty,check_cancellation_fee
3706,"I cannot find the termination penalties, I nee...",check_cancellation_fee
3101,I don't know how I could check the early exit ...,check_cancellation_fee



CATEGORY: FEEDBACK


,instruction,intent
22029,want help to send feedback about a product,review
22172,ya have a method to send a comment about a pro...,review
7398,I'm very unhappyt with ur service i need to fi...,complaint
7184,I have to make a claim against your organization,complaint
7200,where do I make a consumer claim against your ...,complaint
21977,I need help leaving a review for a service,review
7667,"I would like to make a reclamation, how do I d...",complaint
7005,I call to make a customer claim,complaint
22290,I don't know how to leave a comment about a pr...,review
6984,"my experience with you was awful, file a recla...",complaint



CATEGORY: DELIVERY


,instruction,intent
13033,need help to see when my purchase is gonna arrive,delivery_period
13181,wanna check how soon can i epect my delivery h...,delivery_period
12385,"I have to see the shipping options, could I ge...",delivery_options
12197,I have a problem with the shipment options,delivery_options
12214,can you help me see what methods for delivery ...,delivery_options
12982,i want help to check when wqill my package arrive,delivery_period
12645,can I place a bloody order from {{Delivery Cit...,delivery_options
12010,can I order from {{Delivery City}}?,delivery_options
13281,assistance to check how soon can I expect my p...,delivery_period
11979,do you deliver to {{Delivery Country}}?,delivery_options


In [96]:
# Inspect intent distribution across all categories

intent_distribution = (
    train_df
    .groupby(["category", "intent"])
    .size()
    .reset_index(name="count")
    .sort_values(
        ["category", "count"],
        ascending=[True, False]
    )
)

display(intent_distribution)

,category,intent,count
3,ACCOUNT,recover_password,705
4,ACCOUNT,registration_problems,696
1,ACCOUNT,delete_account,692
2,ACCOUNT,edit_account,679
0,ACCOUNT,create_account,673
5,ACCOUNT,switch_account,661
6,CANCEL,check_cancellation_fee,671
7,CONTACT,contact_customer_service,724
8,CONTACT,contact_human_agent,687
9,DELIVERY,delivery_options,725


In [97]:
# Check the relationship between categories and intents

intent_category_check = (
    df.groupby("intent")["category"]
    .nunique()
    .reset_index(name="number_of_categories")
)

conflicting_intents = intent_category_check[
    intent_category_check["number_of_categories"] > 1
]

print("Total unique intents:", df["intent"].nunique())

print(
    "Intents assigned to multiple categories:",
    len(conflicting_intents)
)

display(conflicting_intents)

# Display the complete category-intent mapping
category_intent_mapping = (
    df.groupby(["category", "intent"])
    .size()
    .reset_index(name="count")
    .sort_values(["category", "intent"])
)

display(category_intent_mapping)

Total unique intents: 27
Intents assigned to multiple categories: 0


,intent,number_of_categories


,category,intent,count
0,ACCOUNT,create_account,997
1,ACCOUNT,delete_account,995
2,ACCOUNT,edit_account,1000
3,ACCOUNT,recover_password,995
4,ACCOUNT,registration_problems,999
5,ACCOUNT,switch_account,1000
6,CANCEL,check_cancellation_fee,950
7,CONTACT,contact_customer_service,1000
8,CONTACT,contact_human_agent,999
9,DELIVERY,delivery_options,995


In [98]:
# Detailed review of generalization errors

error_review = challenge_df.loc[
    ~challenge_df["correct"],
    ["instruction", "expected_category", "predicted_category"]
].copy()

error_review["dataset_category_definition"] = error_review[
    "expected_category"
].map({
    "ORDER": "Includes cancel_order, change_order, place_order, track_order",
    "CANCEL": "Only includes check_cancellation_fee",
    "DELIVERY": "Includes delivery_options, delivery_period",
    "FEEDBACK": "Includes complaint, review"
})

display(error_review)

,instruction,expected_category,predicted_category,dataset_category_definition
0,Could you tell me where my package is?,ORDER,DELIVERY,"Includes cancel_order, change_order, place_ord..."
1,I want to track my order,ORDER,REFUND,"Includes cancel_order, change_order, place_ord..."
12,I want to cancel my purchase,CANCEL,ORDER,Only includes check_cancellation_fee
13,How can I cancel this order?,CANCEL,ORDER,Only includes check_cancellation_fee
16,My package has not arrived yet,DELIVERY,ACCOUNT,"Includes delivery_options, delivery_period"
18,The product arrived damaged,FEEDBACK,ORDER,"Includes complaint, review"
19,I would like to share my experience,FEEDBACK,ACCOUNT,"Includes complaint, review"


In [99]:
# Correct manually assigned labels based on dataset taxonomy

challenge_df.loc[
    challenge_df["instruction"].isin([
        "I want to cancel my purchase",
        "How can I cancel this order?"
    ]),
    "expected_category"
] = "ORDER"

# Recalculate correctness
challenge_df["correct"] = (
    challenge_df["expected_category"]
    == challenge_df["predicted_category"]
)

print("Revised Generalization Audit")
print("=" * 40)

print("Total examples:", len(challenge_df))
print("Correct predictions:", challenge_df["correct"].sum())
print("Incorrect predictions:", (~challenge_df["correct"]).sum())

print(
    "Accuracy:",
    round(challenge_df["correct"].mean() * 100, 2),
    "%"
)

display(
    challenge_df.loc[
        ~challenge_df["correct"]
    ]
)

Revised Generalization Audit
Total examples: 22
Correct predictions: 17
Incorrect predictions: 5
Accuracy: 77.27 %


,instruction,expected_category,predicted_category,correct
0,Could you tell me where my package is?,ORDER,DELIVERY,False
1,I want to track my order,ORDER,REFUND,False
16,My package has not arrived yet,DELIVERY,ACCOUNT,False
18,The product arrived damaged,FEEDBACK,ORDER,False
19,I would like to share my experience,FEEDBACK,ACCOUNT,False


In [100]:
# Inspect prediction probabilities for remaining errors

error_cases = challenge_df.loc[
    ~challenge_df["correct"]
].copy()

error_vectors = vectorizer.transform(
    error_cases["instruction"]
)

probabilities = classifier.predict_proba(
    error_vectors
)

classes = classifier.classes_

for index, (_, row) in enumerate(error_cases.iterrows()):

    top_indices = probabilities[index].argsort()[-3:][::-1]

    print("\n" + "=" * 60)
    print("Instruction:", row["instruction"])
    print("Expected:", row["expected_category"])

    for rank, class_index in enumerate(top_indices, start=1):

        print(
            f"{rank}. {classes[class_index]}: "
            f"{probabilities[index][class_index]:.4f}"
        )


Instruction: Could you tell me where my package is?
Expected: ORDER
1. DELIVERY: 0.2997
2. ACCOUNT: 0.1303
3. SHIPPING: 0.1144

Instruction: I want to track my order
Expected: ORDER
1. REFUND: 0.3885
2. ORDER: 0.2558
3. ACCOUNT: 0.0792

Instruction: My package has not arrived yet
Expected: DELIVERY
1. ACCOUNT: 0.3059
2. DELIVERY: 0.1423
3. REFUND: 0.1048

Instruction: The product arrived damaged
Expected: FEEDBACK
1. ORDER: 0.2349
2. DELIVERY: 0.1242
3. FEEDBACK: 0.1164

Instruction: I would like to share my experience
Expected: FEEDBACK
1. ACCOUNT: 0.2194
2. FEEDBACK: 0.2189
3. REFUND: 0.1116


In [101]:
# Build a complete probability table for all remaining errors

error_cases = challenge_df.loc[
    ~challenge_df["correct"]
].copy()

error_vectors = vectorizer.transform(
    error_cases["instruction"]
)

probabilities = classifier.predict_proba(error_vectors)

probability_df = pd.DataFrame(
    probabilities,
    columns=classifier.classes_,
    index=error_cases.index
)

probability_df.insert(
    0,
    "expected_category",
    error_cases["expected_category"]
)

probability_df.insert(
    0,
    "instruction",
    error_cases["instruction"]
)

display(
    probability_df.round(4)
)

,instruction,expected_category,ACCOUNT,CANCEL,CONTACT,DELIVERY,FEEDBACK,INVOICE,ORDER,PAYMENT,REFUND,SHIPPING,SUBSCRIPTION
0,Could you tell me where my package is?,ORDER,0.1303,0.0307,0.0740,0.2997,0.0573,0.0523,0.0524,0.0476,0.0816,0.1144,0.0596
1,I want to track my order,ORDER,0.0792,0.0152,0.0291,0.0704,0.0376,0.0327,0.2558,0.0229,0.3885,0.0400,0.0285
16,My package has not arrived yet,DELIVERY,0.3059,0.0301,0.0528,0.1423,0.0742,0.0701,0.0569,0.0367,0.1048,0.0916,0.0346
18,The product arrived damaged,FEEDBACK,0.1153,0.1074,0.0304,0.1242,0.1164,0.0562,0.2349,0.0402,0.0521,0.0694,0.0535
19,I would like to share my experience,FEEDBACK,0.2194,0.0269,0.0667,0.0677,0.2189,0.0634,0.0749,0.0349,0.1116,0.0752,0.0403


In [102]:
from sklearn.metrics.pairwise import cosine_similarity

# Select the remaining incorrect examples
error_cases = challenge_df.loc[
    ~challenge_df["correct"]
].copy()

# Convert new examples using the fitted vectorizer
error_vectors = vectorizer.transform(
    error_cases["instruction"]
)

# Compare each example with training examples
similarities = cosine_similarity(
    error_vectors,
    X_train_tfidf
)

# Display the five most similar training examples
for row_position, (_, error_row) in enumerate(error_cases.iterrows()):

    print("\n" + "=" * 75)
    print("NEW INSTRUCTION:", error_row["instruction"])
    print("EXPECTED:", error_row["expected_category"])
    print("PREDICTED:", error_row["predicted_category"])
    print("=" * 75)

    top_indices = similarities[row_position].argsort()[-5:][::-1]

    nearest_examples = train_df.iloc[top_indices][
        ["instruction", "category", "intent"]
    ].copy()

    nearest_examples["similarity"] = similarities[
        row_position, top_indices
    ]

    display(nearest_examples.reset_index(drop=True))


NEW INSTRUCTION: Could you tell me where my package is?
EXPECTED: ORDER
PREDICTED: DELIVERY


,instruction,category,intent,similarity
0,could you tell me about the newsletter subscri...,SUBSCRIPTION,newsletter_subscription,0.488118
1,could you tell me about shipping periods?,DELIVERY,delivery_period,0.465739
2,could you tell me about updating my delivery a...,SHIPPING,change_shipping_address,0.464544
3,could you tell me about correcting my shipping...,SHIPPING,change_shipping_address,0.463243
4,could you tell me about changing my delivery a...,SHIPPING,change_shipping_address,0.460337



NEW INSTRUCTION: I want to track my order
EXPECTED: ORDER
PREDICTED: REFUND


,instruction,category,intent,similarity
0,I want to track my restitution,REFUND,track_refund,0.721121
1,"I want to track my rebate, I need assistance",REFUND,track_refund,0.607550
2,"I want to track my restitution, how do I do it?",REFUND,track_refund,0.592404
3,"I need to track my compensation, help me",REFUND,track_refund,0.588153
4,i have got to track my refund,REFUND,track_refund,0.573627



NEW INSTRUCTION: My package has not arrived yet
EXPECTED: DELIVERY
PREDICTED: ACCOUNT


,instruction,category,intent,similarity
0,need to check how soon can i ezpect my package,DELIVERY,delivery_period,0.379294
1,i want help to check when wqill my package arrive,DELIVERY,delivery_period,0.347582
2,I do not knoe how I can see when will my packa...,DELIVERY,delivery_period,0.340927
3,can I check how soon can I expect my package?,DELIVERY,delivery_period,0.331758
4,I need assistance to check how soon can I expe...,DELIVERY,delivery_period,0.320135



NEW INSTRUCTION: The product arrived damaged
EXPECTED: FEEDBACK
PREDICTED: ORDER


,instruction,category,intent,similarity
0,help seeing how soon can I expect the product,DELIVERY,delivery_period,0.489418
1,can uhelp me see how sooj can i expect the pro...,DELIVERY,delivery_period,0.488432
2,where can I check how soon can I expect the pr...,DELIVERY,delivery_period,0.474590
3,help me seeing how soon can i expect the product,DELIVERY,delivery_period,0.472721
4,i have got to see how soon can i expect the pr...,DELIVERY,delivery_period,0.472602



NEW INSTRUCTION: I would like to share my experience
EXPECTED: FEEDBACK
PREDICTED: ACCOUNT


,instruction,category,intent,similarity
0,I would like to cancel purchase {{Order Number}},ORDER,cancel_order,0.493241
1,I would like to cancel purchase {{Order Number}},ORDER,cancel_order,0.493241
2,I would like to cancel purchase {{Order Number}},ORDER,cancel_order,0.493241
3,I would like to cancel purchase {{Order Number}},ORDER,cancel_order,0.493241
4,I would like to request a refund,REFUND,get_refund,0.491405


In [103]:
# Analyze vocabulary coverage for challenge examples

feature_names = set(vectorizer.get_feature_names_out())

coverage_results = []

for _, row in challenge_df.iterrows():

    words = vectorizer.build_analyzer()(
        row["instruction"]
    )

    known_words = [
        word for word in words
        if word in feature_names
    ]

    unknown_words = [
        word for word in words
        if word not in feature_names
    ]

    coverage = (
        len(known_words) / len(words) * 100
        if words else 0
    )

    coverage_results.append({
        "instruction": row["instruction"],
        "known_features": len(known_words),
        "unknown_features": len(unknown_words),
        "coverage_percentage": round(coverage, 2),
        "unknown_words": unknown_words
    })

coverage_df = pd.DataFrame(coverage_results)

display(coverage_df)

,instruction,known_features,unknown_features,coverage_percentage,unknown_words
0,Could you tell me where my package is?,13,2,86.67,"[me where, where my]"
1,I want to track my order,9,0,100.00,[]
2,Please change the address for my delivery,10,3,76.92,"[please, please change, address for]"
3,The shipping address needs to be corrected,9,4,69.23,"[corrected, address needs, needs to, be correc..."
4,My card payment was declined,5,4,55.56,"[declined, card payment, payment was, was decl..."
5,What payment options do you accept?,10,1,90.91,[options do]
6,When will I receive my money back?,10,1,90.91,[will receive]
7,Can I get a refund for this purchase?,10,1,90.91,[refund for]
8,I need to speak with a real person,8,3,72.73,"[real, with real, real person]"
9,How can I reach your support team?,7,4,63.64,"[team, reach your, your support, support team]"


In [104]:
# Identify the most frequently missing features

from collections import Counter

unknown_feature_counts = Counter()

for features in coverage_df["unknown_words"]:
    unknown_feature_counts.update(features)

unknown_features_summary = pd.DataFrame(
    unknown_feature_counts.most_common(),
    columns=["unknown_feature", "frequency"]
)

display(unknown_features_summary)

,unknown_feature,frequency
0,please,2
1,arrived,2
2,me where,1
3,where my,1
4,please change,1
5,address for,1
6,corrected,1
7,address needs,1
8,needs to,1
9,be corrected,1


In [106]:
# Character-level TF-IDF experiment

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score
)

char_tfidf = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_char = char_tfidf.fit_transform(X_train)
y_val_pred_char = char_model.predict(X_val_char)

char_model = LogisticRegression(
    C=1,
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

char_model.fit(X_train_char, y_train)

y_val_pred_char = char_model.predict(X_val)

char_results = {
    "Model": "Character TF-IDF + Logistic Regression",
    "Accuracy": accuracy_score(y_val, y_val_pred_char),
    "Macro F1": f1_score(
        y_val,
        y_val_pred_char,
        average="macro"
    ),
    "Weighted F1": f1_score(
        y_val,
        y_val_pred_char,
        average="weighted"
    )
}

display(pd.DataFrame([char_results]))

ValueError: could not convert string to float: 'question about cancelling order {{Order Number}}'

In [107]:


# Character-level TF-IDF experiment

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score
)

# 1. Create Character TF-IDF vectorizer
char_tfidf = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

# 2. Transform training and validation data
char_tfidf.fit(X_train)

X_train_char = char_tfidf.transform(X_train)
X_val_char = char_tfidf.transform(X_val)

print("Training shape:", X_train_char.shape)
print("Validation shape:", X_val_char.shape)
print("Number of character features:", len(char_tfidf.vocabulary_))


# 3. Train Logistic Regression
char_model = LogisticRegression(
    C=1,
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

char_model.fit(X_train_char, y_train)


# 4. Predict validation categories
y_val_pred_char = char_model.predict(X_val_char)


# 5. Evaluate the model
char_results = {
    "Model": "Character TF-IDF + Logistic Regression",
    "Accuracy": accuracy_score(y_val, y_val_pred_char),
    "Macro F1": f1_score(
        y_val,
        y_val_pred_char,
        average="macro"
    ),
    "Weighted F1": f1_score(
        y_val,
        y_val_pred_char,
        average="weighted"
    )
}

display(pd.DataFrame([char_results]))

Training shape: (18798, 9166)
Validation shape: (4041, 9166)
Number of character features: 9166


,Model,Accuracy,Macro F1,Weighted F1
0,Character TF-IDF + Logistic Regression,0.99901,0.999139,0.99901


In [108]:
# Inspect Character TF-IDF validation errors

char_errors = val_df.copy()

char_errors["predicted_category"] = y_val_pred_char

char_errors["correct"] = (
    char_errors["category"]
    == char_errors["predicted_category"]
)

char_errors = char_errors.loc[
    ~char_errors["correct"]
].copy()

print("Number of validation errors:", len(char_errors))

display(
    char_errors[
        [
            "instruction",
            "category",
            "predicted_category",
            "intent"
        ]
    ]
)

Number of validation errors: 4


,instruction,category,predicted_category,intent
166,would you give me information about cancelling...,ORDER,ACCOUNT,cancel_order
5080,do ya accept card,PAYMENT,ACCOUNT,check_payment_methods
8440,ucstomer service mail address,CONTACT,SHIPPING,contact_customer_service
25589,i have to see the damn ETA of my delivery helpme,ORDER,DELIVERY,track_order


In [109]:
# Compare Word and Character TF-IDF validation errors

comparison_df = val_df[
    ["instruction", "category", "intent"]
].copy()

comparison_df["word_prediction"] = y_val_pred_balanced

comparison_df["char_prediction"] = y_val_pred_char

comparison_df["word_correct"] = (
    comparison_df["word_prediction"]
    == comparison_df["category"]
)

comparison_df["char_correct"] = (
    comparison_df["char_prediction"]
    == comparison_df["category"]
)

comparison_df["error_status"] = comparison_df.apply(
    lambda row: (
        "Both correct"
        if row["word_correct"] and row["char_correct"]
        else "Word only correct"
        if row["word_correct"]
        else "Character only correct"
        if row["char_correct"]
        else "Both wrong"
    ),
    axis=1
)

display(
    comparison_df["error_status"].value_counts()
)

display(
    comparison_df.loc[
        comparison_df["error_status"] != "Both correct",
        [
            "instruction",
            "category",
            "word_prediction",
            "char_prediction",
            "error_status"
        ]
    ]
)

error_status
Both correct              4024
Character only correct      13
Both wrong                   3
Word only correct            1
Name: count, dtype: int64

,instruction,category,word_prediction,char_prediction,error_status
166,would you give me information about cancelling...,ORDER,ACCOUNT,ACCOUNT,Both wrong
1758,correct order370795561790,ORDER,ACCOUNT,ORDER,Character only correct
2252,I can't modify theaddress,SHIPPING,ACCOUNT,SHIPPING,Character only correct
2473,i dont know what i need to do to correct thead...,SHIPPING,ACCOUNT,SHIPPING,Character only correct
2825,I'd like to correct the bloody dadress can i g...,SHIPPING,ACCOUNT,SHIPPING,Character only correct
5080,do ya accept card,PAYMENT,PAYMENT,ACCOUNT,Word only correct
5477,is it possible to check the allowed paymentmod...,PAYMENT,DELIVERY,PAYMENT,Character only correct
6360,are charges for services reimbursable?,REFUND,FEEDBACK,REFUND,Character only correct
6604,are the service charges refundable?,REFUND,CANCEL,REFUND,Character only correct
6813,are charges for services reimbursable,REFUND,FEEDBACK,REFUND,Character only correct


In [110]:
# Compare vocabulary sizes

print("Word TF-IDF features:", len(tfidf.vocabulary_))

print("Character TF-IDF features:", len(char_tfidf.vocabulary_))

print(
    "Combined features:",
    len(tfidf.vocabulary_) + len(char_tfidf.vocabulary_)
)

Word TF-IDF features: 4876
Character TF-IDF features: 9166
Combined features: 14042


In [111]:
# Combined Word + Character TF-IDF experiment

from sklearn.pipeline import FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score
)

# 1. Create combined feature extraction

combined_tfidf = FeatureUnion([
    (
        "word",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True
        )
    ),
    (
        "char",
        TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True
        )
    )
])


# 2. Fit on training data only

X_train_combined = combined_tfidf.fit_transform(X_train)

X_val_combined = combined_tfidf.transform(X_val)

print("Training shape:", X_train_combined.shape)
print("Validation shape:", X_val_combined.shape)


# 3. Train Logistic Regression

combined_model = LogisticRegression(
    C=1,
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

combined_model.fit(X_train_combined, y_train)


# 4. Predict validation categories

y_val_pred_combined = combined_model.predict(X_val_combined)


# 5. Evaluate the combined model

combined_results = {
    "Model": "Word + Character TF-IDF",
    "Accuracy": accuracy_score(
        y_val,
        y_val_pred_combined
    ),
    "Macro F1": f1_score(
        y_val,
        y_val_pred_combined,
        average="macro"
    ),
    "Weighted F1": f1_score(
        y_val,
        y_val_pred_combined,
        average="weighted"
    )
}

display(pd.DataFrame([combined_results]))

Training shape: (18798, 14042)
Validation shape: (4041, 14042)


,Model,Accuracy,Macro F1,Weighted F1
0,Word + Character TF-IDF,0.99901,0.999139,0.99901


In [112]:
# Compare Character and Combined model predictions

model_comparison = val_df[
    ["instruction", "category", "intent"]
].copy()

model_comparison["char_prediction"] = y_val_pred_char

model_comparison["combined_prediction"] = y_val_pred_combined

model_comparison["char_correct"] = (
    model_comparison["char_prediction"]
    == model_comparison["category"]
)

model_comparison["combined_correct"] = (
    model_comparison["combined_prediction"]
    == model_comparison["category"]
)

model_comparison["comparison"] = model_comparison.apply(
    lambda row: (
        "Both correct"
        if row["char_correct"] and row["combined_correct"]
        else "Character only correct"
        if row["char_correct"]
        else "Combined only correct"
        if row["combined_correct"]
        else "Both wrong"
    ),
    axis=1
)

display(model_comparison["comparison"].value_counts())

display(
    model_comparison.loc[
        model_comparison["comparison"] != "Both correct",
        [
            "instruction",
            "category",
            "char_prediction",
            "combined_prediction",
            "comparison"
        ]
    ]
)

comparison
Both correct    4037
Both wrong         4
Name: count, dtype: int64

,instruction,category,char_prediction,combined_prediction,comparison
166,would you give me information about cancelling...,ORDER,ACCOUNT,ACCOUNT,Both wrong
5080,do ya accept card,PAYMENT,ACCOUNT,ACCOUNT,Both wrong
8440,ucstomer service mail address,CONTACT,SHIPPING,SHIPPING,Both wrong
25589,i have to see the damn ETA of my delivery helpme,ORDER,DELIVERY,DELIVERY,Both wrong


In [113]:
# Character TF-IDF + Linear SVM experiment

from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    f1_score
)

# 1. Create Linear SVM model

char_svm_model = LinearSVC(
    C=1,
    class_weight="balanced",
    random_state=42,
    max_iter=5000
)

# 2. Train the model

char_svm_model.fit(X_train_char, y_train)

# 3. Predict validation categories

y_val_pred_svm = char_svm_model.predict(X_val_char)

# 4. Evaluate the model

svm_results = {
    "Model": "Character TF-IDF + Linear SVM",
    "Accuracy": accuracy_score(
        y_val,
        y_val_pred_svm
    ),
    "Macro F1": f1_score(
        y_val,
        y_val_pred_svm,
        average="macro"
    ),
    "Weighted F1": f1_score(
        y_val,
        y_val_pred_svm,
        average="weighted"
    )
}

display(pd.DataFrame([svm_results]))

,Model,Accuracy,Macro F1,Weighted F1
0,Character TF-IDF + Linear SVM,0.999505,0.999571,0.999505


In [114]:
# Inspect Character TF-IDF + Linear SVM errors

svm_errors = val_df.copy()

svm_errors["predicted_category"] = y_val_pred_svm

svm_errors["correct"] = (
    svm_errors["category"]
    == svm_errors["predicted_category"]
)

svm_errors = svm_errors.loc[
    ~svm_errors["correct"]
].copy()

print("Number of SVM validation errors:", len(svm_errors))

display(
    svm_errors[
        [
            "instruction",
            "category",
            "predicted_category",
            "intent"
        ]
    ]
)

Number of SVM validation errors: 2


,instruction,category,predicted_category,intent
5080,do ya accept card,PAYMENT,ACCOUNT,check_payment_methods
25589,i have to see the damn ETA of my delivery helpme,ORDER,DELIVERY,track_order


In [115]:
# Detailed classification report for Character TF-IDF + Linear SVM

from sklearn.metrics import classification_report

svm_report = classification_report(
    y_val,
    y_val_pred_svm,
    output_dict=True,
    zero_division=0
)

svm_report_df = pd.DataFrame(svm_report).T

display(
    svm_report_df[
        ["precision", "recall", "f1-score", "support"]
    ].round(6)
)

,precision,recall,f1-score,support
ACCOUNT,0.998982,1.000000,0.999491,981.000000
CANCEL,1.000000,1.000000,1.000000,150.000000
CONTACT,1.000000,1.000000,1.000000,295.000000
DELIVERY,0.996552,1.000000,0.998273,289.000000
FEEDBACK,1.000000,1.000000,1.000000,275.000000
INVOICE,1.000000,1.000000,1.000000,271.000000
ORDER,1.000000,0.998255,0.999127,573.000000
PAYMENT,1.000000,0.996795,0.998395,312.000000
REFUND,1.000000,1.000000,1.000000,449.000000
SHIPPING,1.000000,1.000000,1.000000,299.000000


In [116]:
# Character TF-IDF + Linear SVM model information

print("Model: Character TF-IDF + Linear SVM")

print(
    "Number of features:",
    X_train_char.shape[1]
)

print(
    "Number of training samples:",
    X_train_char.shape[0]
)

print(
    "Number of support vectors:",
    "Not applicable to LinearSVC in the same way as kernel SVM"
)

print(
    "Number of classes:",
    len(char_svm_model.classes_)
)

print(
    "Regularization parameter (C):",
    char_svm_model.C
)

print(
    "Class weight:",
    char_svm_model.class_weight
)

print(
    "Maximum iterations:",
    char_svm_model.max_iter
)

print(
    "Number of iterations used:",
    char_svm_model.n_iter_
)

Model: Character TF-IDF + Linear SVM
Number of features: 9166
Number of training samples: 18798
Number of support vectors: Not applicable to LinearSVC in the same way as kernel SVM
Number of classes: 11
Regularization parameter (C): 1
Class weight: balanced
Maximum iterations: 5000
Number of iterations used: 12


In [117]:

# Save selected SVM candidate configuration

import json
from pathlib import Path

PROJECT_ROOT = Path.home() / "Desktop" / "Lumora"

CONFIG_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "selected_svm_config_v1.json"
)

svm_config = {
    "model_name": "Character TF-IDF + Linear SVM",
    "version": "1.0",
    "status": "selected_candidate_pending_test_evaluation",

    "vectorizer": {
        "analyzer": "char_wb",
        "ngram_range": [3, 5],
        "min_df": 2,
        "max_df": 0.95,
        "sublinear_tf": True
    },

    "classifier": {
        "type": "LinearSVC",
        "C": 1,
        "class_weight": "balanced",
        "random_state": 42,
        "max_iter": 5000
    },

    "validation": {
        "accuracy": 0.999505,
        "macro_f1": 0.999571,
        "weighted_f1": 0.999505,
        "errors": 2
    },

    "training_samples": 18798,
    "number_of_features": 9166,
    "number_of_classes": 11
}

CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(CONFIG_PATH, "w") as file:
    json.dump(svm_config, file, indent=4)

print("Configuration saved successfully!")
print("Path:", CONFIG_PATH)

Configuration saved successfully!
Path: /Users/raghad/Desktop/Lumora/data/processed/selected_svm_config_v1.json


In [118]:
# Prepare Test data for the selected Character SVM model

X_test_char = char_tfidf.transform(X_test)

print("Test TF-IDF shape:", X_test_char.shape)
print("Number of test samples:", X_test_char.shape[0])
print("Number of features:", X_test_char.shape[1])

Test TF-IDF shape: (4033, 9166)
Number of test samples: 4033
Number of features: 9166


In [119]:
# Final Test Evaluation - Character TF-IDF + Linear SVM

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Generate predictions
y_test_pred_svm = char_svm_model.predict(X_test_char)

# Calculate evaluation metrics
svm_test_accuracy = accuracy_score(
    y_test,
    y_test_pred_svm
)

svm_test_macro_f1 = f1_score(
    y_test,
    y_test_pred_svm,
    average="macro"
)

svm_test_weighted_f1 = f1_score(
    y_test,
    y_test_pred_svm,
    average="weighted"
)

# Display results
print("Character TF-IDF + Linear SVM")
print("-" * 45)

print(f"Test Accuracy: {svm_test_accuracy:.6f}")
print(f"Test Macro F1: {svm_test_macro_f1:.6f}")
print(f"Test Weighted F1: {svm_test_weighted_f1:.6f}")

print(
    "Number of errors:",
    (y_test != y_test_pred_svm).sum()
)

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_test_pred_svm,
        zero_division=0
    )
)

Character TF-IDF + Linear SVM
---------------------------------------------
Test Accuracy: 0.999752
Test Macro F1: 0.999576
Test Weighted F1: 0.999752
Number of errors: 1

Classification Report:
              precision    recall  f1-score   support

     ACCOUNT       1.00      1.00      1.00       899
      CANCEL       0.99      1.00      1.00       129
     CONTACT       1.00      1.00      1.00       293
    DELIVERY       1.00      1.00      1.00       289
    FEEDBACK       1.00      1.00      1.00       309
     INVOICE       1.00      1.00      1.00       303
       ORDER       1.00      1.00      1.00       620
     PAYMENT       1.00      1.00      1.00       290
      REFUND       1.00      1.00      1.00       475
    SHIPPING       1.00      1.00      1.00       287
SUBSCRIPTION       1.00      1.00      1.00       139

    accuracy                           1.00      4033
   macro avg       1.00      1.00      1.00      4033
weighted avg       1.00      1.00      1.00    

In [120]:
# Inspect the single SVM Test error

svm_test_errors = test_df.copy()

svm_test_errors["predicted_category"] = y_test_pred_svm

svm_test_errors["correct"] = (
    svm_test_errors["category"]
    == svm_test_errors["predicted_category"]
)

svm_test_errors = svm_test_errors.loc[
    ~svm_test_errors["correct"]
].copy()

print("Number of Test errors:", len(svm_test_errors))

display(
    svm_test_errors[
        [
            "instruction",
            "category",
            "predicted_category",
            "intent"
        ]
    ]
)

Number of Test errors: 1


,instruction,category,predicted_category,intent
952,order cancellation,ORDER,CANCEL,cancel_order


In [121]:
char_tfidf
char_svm_model

,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to ``class_weight[i]*C`` forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",'balanced'
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo random number generation for shuffling the data forthe dual coordinate descent (if ``dual=True``). When ``dual=False`` theunderlying implementation of :class:`LinearSVC` is not random and``random_state`` has no effect on the results.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"max_iter max_iter: int, default=1000The maximum number of iterations to be run.",5000
,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_

In [122]:
print("Vectorizer fitted:", hasattr(char_tfidf, "vocabulary_"))
print("SVM fitted:", hasattr(char_svm_model, "classes_"))

print("Number of features:", len(char_tfidf.vocabulary_))
print("Number of classes:", len(char_svm_model.classes_))

Vectorizer fitted: True
SVM fitted: True
Number of features: 9166
Number of classes: 11


In [123]:
import joblib
from pathlib import Path

PROJECT_ROOT = Path.home() / "Desktop" / "Lumora"

MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "lumora_model_v2.joblib"
)

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

model_artifact = {
    "vectorizer": char_tfidf,
    "classifier": char_svm_model,
    "categories": char_svm_model.classes_.tolist(),
    "model_name": "Character TF-IDF + Linear SVM",
    "version": "2.0",
    "metrics": {
        "validation_accuracy": 0.999505,
        "validation_macro_f1": 0.999571,
        "test_accuracy": 0.999752,
        "test_macro_f1": 0.999576,
        "test_errors": 1
    }
}

joblib.dump(model_artifact, MODEL_PATH)

print("Lumora Model V2 saved successfully!")
print("Location:", MODEL_PATH)
print(f"File size: {MODEL_PATH.stat().st_size / 1024:.2f} KB")

Lumora Model V2 saved successfully!
Location: /Users/raghad/Desktop/Lumora/models/lumora_model_v2.joblib
File size: 951.44 KB


In [124]:
from pathlib import Path
from functools import lru_cache

import joblib


PROJECT_ROOT = Path(__file__).resolve().parent.parent

MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "lumora_model_v2.joblib"
)


@lru_cache(maxsize=1)
def load_model():
    """
    Load and cache the trained Lumora model artifact.
    """

    if not MODEL_PATH.exists():
        raise FileNotFoundError(
            f"Model file not found: {MODEL_PATH}"
        )

    artifact = joblib.load(MODEL_PATH)

    return artifact


def predict_category(text):
    """
    Predict the category of a customer support ticket.
    """

    if not isinstance(text, str):
        raise TypeError("Input must be a string.")

    text = text.strip()

    if not text:
        raise ValueError("Input text cannot be empty.")

    artifact = load_model()

    vectorizer = artifact["vectorizer"]
    classifier = artifact["classifier"]

    text_vector = vectorizer.transform([text])

    prediction = classifier.predict(text_vector)[0]

    return prediction

NameError: name '__file__' is not defined

# Intent Classification

Training and evaluating a dedicated model to predict customer intent.

In [125]:
# Prepare data for Intent Classification

X_train_intent = train_df["instruction"].astype(str)
y_train_intent = train_df["intent"]

X_val_intent = val_df["instruction"].astype(str)
y_val_intent = val_df["intent"]

print("Training samples:", len(X_train_intent))
print("Validation samples:", len(X_val_intent))

print("\nNumber of unique intents:", y_train_intent.nunique())

print("\nIntent distribution in training data:")
display(y_train_intent.value_counts().to_frame("Count"))

Training samples: 18798
Validation samples: 4041

Number of unique intents: 27

Intent distribution in training data:


,Count
intent,
delivery_options,725
contact_customer_service,724
check_invoice,722
change_order,720
newsletter_subscription,713
complaint,712
recover_password,705
payment_issue,704
get_invoice,703


In [126]:
# Intent Classification: Character TF-IDF + Linear SVM

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report

intent_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_intent_vec = intent_vectorizer.fit_transform(X_train_intent)
X_val_intent_vec = intent_vectorizer.transform(X_val_intent)

intent_model = LinearSVC(
    C=1,
    random_state=42,
    max_iter=5000
)

intent_model.fit(X_train_intent_vec, y_train_intent)

y_val_intent_pred = intent_model.predict(X_val_intent_vec)

print("Intent Model Evaluation")
print("-" * 40)

print("Accuracy:", accuracy_score(y_val_intent, y_val_intent_pred))
print("Macro F1:", f1_score(y_val_intent, y_val_intent_pred, average="macro"))
print("Weighted F1:", f1_score(y_val_intent, y_val_intent_pred, average="weighted"))

print("\nClassification Report:")
print(classification_report(y_val_intent, y_val_intent_pred))

print("Number of features:", len(intent_vectorizer.get_feature_names_out()))

Intent Model Evaluation
----------------------------------------
Accuracy: 0.9987626825043306
Macro F1: 0.9987305680239265
Weighted F1: 0.9987627128733769

Classification Report:
                          precision    recall  f1-score   support

            cancel_order       1.00      1.00      1.00       163
            change_order       0.99      0.99      0.99       125
 change_shipping_address       0.99      1.00      1.00       144
  check_cancellation_fee       1.00      1.00      1.00       150
           check_invoice       1.00      1.00      1.00       130
   check_payment_methods       1.00      1.00      1.00       153
     check_refund_policy       1.00      0.99      1.00       160
               complaint       1.00      1.00      1.00       137
contact_customer_service       1.00      1.00      1.00       149
     contact_human_agent       1.00      1.00      1.00       146
          create_account       0.99      1.00      1.00       186
          delete_account    

In [127]:
# Inspect Intent Classification Errors

intent_errors = pd.DataFrame({
    "instruction": X_val_intent,
    "true_intent": y_val_intent,
    "predicted_intent": y_val_intent_pred
})

intent_errors = intent_errors[
    intent_errors["true_intent"] != intent_errors["predicted_intent"]
]

print("Total errors:", len(intent_errors))

display(intent_errors.reset_index(drop=True))

Total errors: 5


,instruction,true_intent,predicted_intent
0,could you help me to delete something?,change_order,place_order
1,am I entitled to a reimbursement?,check_refund_policy,get_refund
2,assistance opening user account,registration_problems,create_account
3,deliveryaddress issue,set_up_shipping_address,change_shipping_address
4,where is purechase {{Order Number}},track_order,change_order


## Intent Model — Validation Results

- Model: Character TF-IDF + Linear SVM
- Number of intents: 27
- Training samples: 18,798
- Validation samples: 4,041
- Accuracy: 99.8763%
- Macro F1: 99.8731%
- Validation errors: 5
- Feature count: 9,166
- Status: Selected candidate pending test evaluation

In [128]:
# Save Intent Model Candidate

import joblib
import json
from pathlib import Path

PROJECT_ROOT = Path.home() / "Desktop" / "Lumora"

MODEL_DIR = PROJECT_ROOT / "models"
CONFIG_DIR = PROJECT_ROOT / "data" / "processed"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

intent_artifact = {
    "vectorizer": intent_vectorizer,
    "classifier": intent_model,
    "intents": intent_model.classes_.tolist(),
    "model_name": "Character TF-IDF + Linear SVM",
    "version": "1.0",
    "metrics": {
        "validation_accuracy": accuracy_score(
            y_val_intent, y_val_intent_pred
        ),
        "validation_macro_f1": f1_score(
            y_val_intent, y_val_intent_pred, average="macro"
        ),
        "validation_weighted_f1": f1_score(
            y_val_intent, y_val_intent_pred, average="weighted"
        ),
        "validation_errors": len(intent_errors)
    }
}

MODEL_PATH = MODEL_DIR / "lumora_intent_model_v1.joblib"

joblib.dump(intent_artifact, MODEL_PATH)

intent_config = {
    "model_name": "Character TF-IDF + Linear SVM",
    "version": "1.0",
    "status": "selected_candidate_pending_test_evaluation",
    "vectorizer": {
        "analyzer": "char_wb",
        "ngram_range": [3, 5],
        "min_df": 2,
        "max_df": 0.95,
        "sublinear_tf": True
    },
    "classifier": {
        "type": "LinearSVC",
        "C": 1,
        "random_state": 42,
        "max_iter": 5000
    },
    "validation": intent_artifact["metrics"],
    "training_samples": len(X_train_intent),
    "validation_samples": len(X_val_intent),
    "number_of_features": len(
        intent_vectorizer.get_feature_names_out()
    ),
    "number_of_intents": len(intent_model.classes_)
}

CONFIG_PATH = CONFIG_DIR / "selected_intent_config_v1.json"

with open(CONFIG_PATH, "w") as file:
    json.dump(intent_config, file, indent=4)

print("Intent model saved successfully.")
print("Model:", MODEL_PATH)
print("Config:", CONFIG_PATH)
print("Model size:", round(MODEL_PATH.stat().st_size / 1024, 2), "KB")

Intent model saved successfully.
Model: /Users/raghad/Desktop/Lumora/models/lumora_intent_model_v1.joblib
Config: /Users/raghad/Desktop/Lumora/data/processed/selected_intent_config_v1.json
Model size: 2098.04 KB


In [129]:
# Test Category and Intent Models Together

from src.predictor import predict_category

test_messages = [
    "I want to track my order",
    "I need help getting a refund",
    "I want to cancel my order",
    "What payment methods do you accept?",
    "I want to change my shipping address"
]

for message in test_messages:
    category = predict_category(message)

    intent_vector = intent_vectorizer.transform([message])
    intent = intent_model.predict(intent_vector)[0]

    print(f"Customer: {message}")
    print(f"Category: {category}")
    print(f"Intent: {intent}")
    print("-" * 50)

ModuleNotFoundError: No module named 'src'

In [130]:
# Test Category and Intent Models Together

import sys
from pathlib import Path

PROJECT_ROOT = Path.home() / "Desktop" / "Lumora"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.predictor import predict_category

test_messages = [
    "I want to track my order",
    "I need help getting a refund",
    "I want to cancel my order",
    "What payment methods do you accept?",
    "I want to change my shipping address"
]

for message in test_messages:
    category = predict_category(message)

    intent_vector = intent_vectorizer.transform([message])
    intent = intent_model.predict(intent_vector)[0]

    print(f"Customer: {message}")
    print(f"Category: {category}")
    print(f"Intent: {intent}")
    print("-" * 50)

Customer: I want to track my order
Category: ORDER
Intent: track_order
--------------------------------------------------
Customer: I need help getting a refund
Category: REFUND
Intent: get_refund
--------------------------------------------------
Customer: I want to cancel my order
Category: ORDER
Intent: cancel_order
--------------------------------------------------
Customer: What payment methods do you accept?
Category: PAYMENT
Intent: check_payment_methods
--------------------------------------------------
Customer: I want to change my shipping address
Category: SHIPPING
Intent: change_shipping_address
--------------------------------------------------


In [131]:
# Test Lumora Intelligence Engine

from src.intelligence import analyze_message

test_messages = [
    "I want to track my order",
    "I need help getting a refund",
    "I want to cancel my order",
    "What payment methods do you accept?",
    "I want to change my shipping address"
]

for message in test_messages:
    result = analyze_message(message)

    print("Customer:", result["message"])
    print("Category:", result["category"])
    print("Intent:", result["intent"])
    print("-" * 50)

Customer: I want to track my order
Category: ORDER
Intent: track_order
--------------------------------------------------
Customer: I need help getting a refund
Category: REFUND
Intent: get_refund
--------------------------------------------------
Customer: I want to cancel my order
Category: ORDER
Intent: cancel_order
--------------------------------------------------
Customer: What payment methods do you accept?
Category: PAYMENT
Intent: check_payment_methods
--------------------------------------------------
Customer: I want to change my shipping address
Category: SHIPPING
Intent: change_shipping_address
--------------------------------------------------


In [132]:
 # Test Knowledge Base Loader

from src.knowledge_base import get_knowledge_documents

documents = get_knowledge_documents()

print("Total documents:", len(documents))
print("\nKnowledge Base Sections:\n")

for index, document in enumerate(documents, start=1):
    print(f"{index}. {document['title']}")
    print("Content length:", len(document["content"]))
    print("-" * 50)

Total documents: 10

Knowledge Base Sections:

1. Version 1.0 — Fictional E-commerce Store
Content length: 218
--------------------------------------------------
2. 1. Store Overview
Content length: 228
--------------------------------------------------
3. 2. Orders
Content length: 795
--------------------------------------------------
4. 3. Shipping and Delivery
Content length: 673
--------------------------------------------------
5. 4. Refunds and Returns
Content length: 910
--------------------------------------------------
6. 5. Payments
Content length: 517
--------------------------------------------------
7. 6. Customer Accounts
Content length: 553
--------------------------------------------------
8. 7. Subscriptions
Content length: 422
--------------------------------------------------
9. 8. Customer Support
Content length: 482
--------------------------------------------------
10. 9. AI Assistant Guidelines
Content length: 888
-------------------------------------------------

In [133]:
from src.knowledge_base import get_knowledge_documents

chunks = get_knowledge_documents()

print("Total chunks:", len(chunks))
print("\nFirst 5 chunks:\n")

for chunk in chunks[:5]:
    print("ID:", chunk["chunk_id"])
    print("Section:", chunk["section"])
    print("Subsection:", chunk["subsection"])
    print("Content:", chunk["content"][:250])
    print("-" * 60)

Total chunks: 10

First 5 chunks:



KeyError: 'chunk_id'

In [134]:
import importlib
import src.knowledge_base as kb

importlib.reload(kb)

chunks = kb.get_knowledge_documents()

print("Total chunks:", len(chunks))
print("\nFirst 5 chunks:\n")

for chunk in chunks[:5]:
    print("ID:", chunk["chunk_id"])
    print("Section:", chunk["section"])
    print("Subsection:", chunk["subsection"])
    print("Content:", chunk["content"][:250])
    print("-" * 60)

Total chunks: 22

First 5 chunks:

ID: chunk_001
Section: 1. Store Overview
Subsection: None
Content: **Store Name:** Lumora Demo Store

**Business Type:** Online Retail Store

**Description:** An online store that allows customers to browse products, place orders, make payments, and manage their accounts and subscriptions.

---
------------------------------------------------------------
ID: chunk_002
Section: 2. Orders
Subsection: 2.1 Order Tracking
Content: - Customers can track their orders using their order number.
- Tracking information becomes available after the order has been dispatched.
- If tracking information is unavailable, customers should contact customer support.
- Lumora cannot access liv
------------------------------------------------------------
ID: chunk_003
Section: 2. Orders
Subsection: 2.2 Order Cancellation
Content: - Customers may request order cancellation before the order is dispatched.
- Cancellation requests after dispatch require assistance from customer

In [135]:
import pandas as pd

chunks_df = pd.DataFrame(chunks)

chunks_df["content_length"] = chunks_df["content"].str.len()

display(
    chunks_df[
        ["chunk_id", "section", "subsection", "content_length"]
    ]
)

print("\nChunk Statistics:")
print(chunks_df["content_length"].describe())

,chunk_id,section,subsection,content_length
0,chunk_001,1. Store Overview,NaN,228
1,chunk_002,2. Orders,2.1 Order Tracking,321
2,chunk_003,2. Orders,2.2 Order Cancellation,218
3,chunk_004,2. Orders,2.3 Order Modification,175
4,chunk_005,3. Shipping and Delivery,3.1 Delivery Options,203
5,chunk_006,3. Shipping and Delivery,3.2 Shipping Address Changes,197
6,chunk_007,3. Shipping and Delivery,3.3 Delivery Delays,187
7,chunk_008,4. Refunds and Returns,4.1 Return Eligibility,264
8,chunk_009,4. Refunds and Returns,4.2 Refund Processing,196
9,chunk_010,4. Refunds and Returns,4.3 Refund Tracking,153



Chunk Statistics:
count     22.000000
mean     222.363636
std      157.262155
min       76.000000
25%      158.500000
50%      189.000000
75%      214.250000
max      888.000000
Name: content_length, dtype: float64


In [136]:
from src.retriever import retrieve_documents

test_queries = [
    "Where is my order?",
    "How long does a refund take?",
    "Can I change my shipping address?",
    "What payment methods do you accept?",
    "I forgot my password. How can I reset it?",
    "I want to speak to a human agent."
]

for query in test_queries:

    print(f"\nCustomer Query: {query}")
    print("-" * 70)

    results = retrieve_documents(query, top_k=3)

    for result in results:
        print(f"Section: {result['section']}")
        print(f"Subsection: {result['subsection']}")
        print(f"Similarity Score: {result['score']}")
        print(f"Content: {result['content'][:200]}")
        print("-" * 50)


Customer Query: Where is my order?
----------------------------------------------------------------------
Section: 2. Orders
Subsection: 2.2 Order Cancellation
Similarity Score: 0.2372
Content: - Customers may request order cancellation before the order is dispatched.
- Cancellation requests after dispatch require assistance from customer support.
- Cancellation is not guaranteed until confi
--------------------------------------------------
Section: 2. Orders
Subsection: 2.3 Order Modification
Similarity Score: 0.1852
Content: - Customers may request changes to an order before it is processed.
- Changes after processing are not guaranteed.
- Customers should contact support as soon as possible.

---
--------------------------------------------------
Section: 2. Orders
Subsection: 2.1 Order Tracking
Similarity Score: 0.1818
Content: - Customers can track their orders using their order number.
- Tracking information becomes available after the order has been dispatched.
- If tracking i

In [137]:
for query in test_queries:

    results = retrieve_documents(query, top_k=3)

    print(f"\nQuery: {query}")

    for rank, result in enumerate(results, start=1):
        print(
            f"{rank}. {result['subsection']} "
            f"| Score: {result['score']}"
        )


Query: Where is my order?
1. 2.2 Order Cancellation | Score: 0.2372
2. 2.3 Order Modification | Score: 0.1852
3. 2.1 Order Tracking | Score: 0.1818

Query: How long does a refund take?
1. 4.2 Refund Processing | Score: 0.1626
2. 4.3 Refund Tracking | Score: 0.1096
3. 4.4 Refund Policy Questions | Score: 0.0978

Query: Can I change my shipping address?
1. 3.2 Shipping Address Changes | Score: 0.4013
2. 3.1 Delivery Options | Score: 0.0925
3. 6.1 Account Registration | Score: 0.0811

Query: What payment methods do you accept?
1. 5.1 Accepted Payment Methods | Score: 0.3713
2. 5.2 Payment Issues | Score: 0.0689
3. 5.3 Payment Security | Score: 0.0623

Query: I forgot my password. How can I reset it?
1. 6.2 Password Recovery | Score: 0.3393
2. 6.1 Account Registration | Score: 0.1033
3. 2.3 Order Modification | Score: 0.067

Query: I want to speak to a human agent.
1. 8.2 Human Agent Escalation | Score: 0.3979
2. 8.1 Contacting Support | Score: 0.0707
3. 6.2 Password Recovery | Score: 0.0

In [138]:
from src.intelligence import analyze_message
from src.retriever import retrieve_documents

for query in test_queries:

    analysis = analyze_message(query)
    results = retrieve_documents(query, top_k=3)

    print(f"\nCustomer Query: {query}")
    print(f"Predicted Category: {analysis['category']}")
    print(f"Predicted Intent: {analysis['intent']}")

    print("\nRetrieved Knowledge:")

    for result in results:
        print(
            f"- {result['subsection']} "
            f"| Score: {result['score']}"
        )

    print("=" * 70)


Customer Query: Where is my order?
Predicted Category: ORDER
Predicted Intent: place_order

Retrieved Knowledge:
- 2.2 Order Cancellation | Score: 0.2372
- 2.3 Order Modification | Score: 0.1852
- 2.1 Order Tracking | Score: 0.1818

Customer Query: How long does a refund take?
Predicted Category: REFUND
Predicted Intent: check_refund_policy

Retrieved Knowledge:
- 4.2 Refund Processing | Score: 0.1626
- 4.3 Refund Tracking | Score: 0.1096
- 4.4 Refund Policy Questions | Score: 0.0978

Customer Query: Can I change my shipping address?
Predicted Category: SHIPPING
Predicted Intent: change_shipping_address

Retrieved Knowledge:
- 3.2 Shipping Address Changes | Score: 0.4013
- 3.1 Delivery Options | Score: 0.0925
- 6.1 Account Registration | Score: 0.0811

Customer Query: What payment methods do you accept?
Predicted Category: PAYMENT
Predicted Intent: check_payment_methods

Retrieved Knowledge:
- 5.1 Accepted Payment Methods | Score: 0.3713
- 5.2 Payment Issues | Score: 0.0689
- 5.3 Paym

In [139]:
print("TRACK_ORDER examples:")
print("-" * 60)

print(
    train_df.loc[
        train_df["intent"] == "track_order",
        "instruction"
    ].sample(10, random_state=42).to_string(index=False)
)

print("\nPLACE_ORDER examples:")
print("-" * 60)

print(
    train_df.loc[
        train_df["intent"] == "place_order",
        "instruction"
    ].sample(10, random_state=42).to_string(index=False)
)

TRACK_ORDER examples:
------------------------------------------------------------
i need help checking the status or purchase {{O...
i have to see the ETA of order {{Order Number}}...
  wanna check the status of order {{Order Number}}
i want help to check the ETA of the oder {{Orde...
             order {{Order Number}} current status
    check the ETA of the purchase {{Order Number}}
          can I see order {{Order Number}} status?
        can ushow me order {{Order Number}} status
         wanna check order {{Order Number}} status
where can i see the ETA of the order {{Order Nu...

PLACE_ORDER examples:
------------------------------------------------------------
                  where can I order some articles?
 I don't know what to do to earn some of your item
   is it possible to acquire several of ur article
I want assistance to acquire several  of your p...
                           how to shop some items?
I would like to earn a product, could I get som...
         i dont k

In [140]:
from src.intelligence import load_intent_model

artifact = load_intent_model()

vectorizer = artifact["vectorizer"]
classifier = artifact["classifier"]

test_messages = [
    "Where is my order?",
    "Where can I find my order?",
    "Can you track my order?",
    "I want to know where my order is",
    "What is the status of my order?",
    "I want to place an order",
    "How can I order a product?"
]

for message in test_messages:

    vector = vectorizer.transform([message])
    scores = classifier.decision_function(vector).flatten()

    top_indices = scores.argsort()[::-1][:3]

    print(f"\nMessage: {message}")
    print("Top predictions:")

    for index in top_indices:
        print(
            f"{classifier.classes_[index]}: "
            f"{scores[index]:.4f}"
        )

    print("-" * 55)


Message: Where is my order?
Top predictions:
place_order: -0.0602
delivery_period: -0.7694
recover_password: -0.8146
-------------------------------------------------------

Message: Where can I find my order?
Top predictions:
place_order: -0.2369
check_invoice: -0.5542
delivery_period: -0.7014
-------------------------------------------------------

Message: Can you track my order?
Top predictions:
track_order: 0.3198
track_refund: -0.5386
place_order: -0.7313
-------------------------------------------------------

Message: I want to know where my order is
Top predictions:
place_order: -0.0213
delivery_period: -0.7207
recover_password: -0.8576
-------------------------------------------------------

Message: What is the status of my order?
Top predictions:
track_order: 0.1489
track_refund: -0.1115
place_order: -0.8226
-------------------------------------------------------

Message: I want to place an order
Top predictions:
place_order: 0.6574
delivery_options: -0.8460
cancel_order:

In [141]:
import pandas as pd

tracking_examples = [
    "I need to know where my package is",
    "Can you check the latest update on my order?",
    "I'm trying to locate my purchase",
    "I haven't received my order yet and need an update",
    "Could you tell me the current status of my purchase?",
    "Where is my package right now?",
    "I want to check the progress of my order",
    "Can you look up my shipment?",
    "I need an update about my order",
    "Has my order been shipped yet?",
    "Please help me find my package",
    "I'd like to know what happened to my order"
]

tracking_augmentation = pd.DataFrame({
    "instruction": tracking_examples,
    "intent": "track_order"
})

display(tracking_augmentation)

,instruction,intent
0,I need to know where my package is,track_order
1,Can you check the latest update on my order?,track_order
2,I'm trying to locate my purchase,track_order
3,I haven't received my order yet and need an up...,track_order
4,Could you tell me the current status of my pur...,track_order
5,Where is my package right now?,track_order
6,I want to check the progress of my order,track_order
7,Can you look up my shipment?,track_order
8,I need an update about my order,track_order
9,Has my order been shipped yet?,track_order


In [142]:
import re
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report


# 1. Normalize text to detect exact duplicates
def normalize_instruction(text):
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


# 2. Check new examples against all existing data splits
existing_texts = set()

for dataframe in [train_df, val_df, test_df]:
    existing_texts.update(
        dataframe["instruction"].astype(str).apply(normalize_instruction)
    )

augmentation_filtered = tracking_augmentation.copy()

augmentation_filtered["normalized"] = (
    augmentation_filtered["instruction"].apply(normalize_instruction)
)

augmentation_filtered = augmentation_filtered[
    ~augmentation_filtered["normalized"].isin(existing_texts)
].drop(columns="normalized")

print("Original new examples:", len(tracking_augmentation))
print("Accepted new examples:", len(augmentation_filtered))
print("Removed duplicates:", len(tracking_augmentation) - len(augmentation_filtered))


# 3. Add accepted examples to training data only
augmented_train = pd.concat(
    [
        train_df[["instruction", "intent"]],
        augmentation_filtered
    ],
    ignore_index=True
)

X_train_candidate = augmented_train["instruction"].astype(str)
y_train_candidate = augmented_train["intent"]

X_val_candidate = val_df["instruction"].astype(str)
y_val_candidate = val_df["intent"]


# 4. Train a new candidate model
candidate_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_vectorized = candidate_vectorizer.fit_transform(
    X_train_candidate
)

X_val_vectorized = candidate_vectorizer.transform(
    X_val_candidate
)

candidate_model = LinearSVC(
    C=1,
    random_state=42,
    max_iter=5000
)

candidate_model.fit(
    X_train_vectorized,
    y_train_candidate
)


# 5. Evaluate on the unchanged validation set
val_predictions_candidate = candidate_model.predict(
    X_val_vectorized
)

candidate_accuracy = accuracy_score(
    y_val_candidate,
    val_predictions_candidate
)

candidate_macro_f1 = f1_score(
    y_val_candidate,
    val_predictions_candidate,
    average="macro"
)

print("\nCandidate Model Results")
print("-" * 45)
print(f"Accuracy: {candidate_accuracy:.4%}")
print(f"Macro F1: {candidate_macro_f1:.4%}")

print("\nClassification Report:")
print(
    classification_report(
        y_val_candidate,
        val_predictions_candidate,
        zero_division=0
    )
)

Original new examples: 12
Accepted new examples: 12
Removed duplicates: 0

Candidate Model Results
---------------------------------------------
Accuracy: 99.8763%
Macro F1: 99.8731%

Classification Report:
                          precision    recall  f1-score   support

            cancel_order       1.00      1.00      1.00       163
            change_order       0.99      0.99      0.99       125
 change_shipping_address       0.99      1.00      1.00       144
  check_cancellation_fee       1.00      1.00      1.00       150
           check_invoice       1.00      1.00      1.00       130
   check_payment_methods       1.00      1.00      1.00       153
     check_refund_policy       1.00      0.99      1.00       160
               complaint       1.00      1.00      1.00       137
contact_customer_service       1.00      1.00      1.00       149
     contact_human_agent       1.00      1.00      1.00       146
          create_account       0.99      1.00      1.00       186


In [143]:
tracking_test_messages = [
    "Where is my order?",
    "Where can I find my order?",
    "Can you track my order?",
    "I want to know where my order is",
    "What is the status of my order?",
    "I need to know where my package is",
    "Can you check the latest update on my order?",
    "I'm trying to locate my purchase",
    "Has my order been shipped yet?",
    "I want to place an order",
    "How can I order a product?"
]

print(f"{'CUSTOMER MESSAGE':<55} {'OLD MODEL':<20} {'NEW MODEL'}")
print("=" * 105)

for message in tracking_test_messages:

    old_prediction = intent_model.predict(
        intent_vectorizer.transform([message])
    )[0]

    new_prediction = candidate_model.predict(
        candidate_vectorizer.transform([message])
    )[0]

    print(
        f"{message[:53]:<55} "
        f"{old_prediction:<20} "
        f"{new_prediction}"
    )

CUSTOMER MESSAGE                                        OLD MODEL            NEW MODEL
Where is my order?                                      place_order          track_order
Where can I find my order?                              place_order          track_order
Can you track my order?                                 track_order          track_order
I want to know where my order is                        place_order          track_order
What is the status of my order?                         track_order          track_order
I need to know where my package is                      delivery_period      track_order
Can you check the latest update on my order?            edit_account         track_order
I'm trying to locate my purchase                        track_order          track_order
Has my order been shipped yet?                          delivery_options     track_order
I want to place an order                                place_order          place_order
How can I order a produ

In [144]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Prepare test data
X_test_intent = test_df["instruction"].astype(str)
y_test_intent = test_df["intent"]

# Original model predictions
old_test_predictions = intent_model.predict(
    intent_vectorizer.transform(X_test_intent)
)

# Candidate model predictions
new_test_predictions = candidate_model.predict(
    candidate_vectorizer.transform(X_test_intent)
)

# Calculate metrics
results = {
    "Original Model": {
        "Accuracy": accuracy_score(
            y_test_intent, old_test_predictions
        ),
        "Macro F1": f1_score(
            y_test_intent, old_test_predictions, average="macro"
        )
    },
    "Candidate Model": {
        "Accuracy": accuracy_score(
            y_test_intent, new_test_predictions
        ),
        "Macro F1": f1_score(
            y_test_intent, new_test_predictions, average="macro"
        )
    }
}

# Display comparison
for model_name, metrics in results.items():
    print(f"\n{model_name}")
    print("-" * 40)
    print(f"Accuracy: {metrics['Accuracy']:.4%}")
    print(f"Macro F1: {metrics['Macro F1']:.4%}")

print("\nCandidate Model Classification Report:")
print(
    classification_report(
        y_test_intent,
        new_test_predictions,
        zero_division=0
    )
)


Original Model
----------------------------------------
Accuracy: 99.7520%
Macro F1: 99.7566%

Candidate Model
----------------------------------------
Accuracy: 99.8264%
Macro F1: 99.8263%

Candidate Model Classification Report:
                          precision    recall  f1-score   support

            cancel_order       1.00      0.99      1.00       142
            change_order       0.99      0.98      0.99       152
 change_shipping_address       1.00      1.00      1.00       141
  check_cancellation_fee       0.99      1.00      1.00       129
           check_invoice       0.99      1.00      1.00       148
   check_payment_methods       1.00      1.00      1.00       154
     check_refund_policy       1.00      1.00      1.00       150
               complaint       1.00      1.00      1.00       151
contact_customer_service       1.00      1.00      1.00       127
     contact_human_agent       1.00      1.00      1.00       166
          create_account       0.99      1

In [145]:
# Compare incorrect predictions from both models

comparison_df = pd.DataFrame({
    "instruction": X_test_intent.values,
    "true_intent": y_test_intent.values,
    "old_prediction": old_test_predictions,
    "new_prediction": new_test_predictions
})

# Show errors made by the candidate model
candidate_errors = comparison_df[
    comparison_df["true_intent"] != comparison_df["new_prediction"]
]

print("Candidate Model Errors:", len(candidate_errors))
print("=" * 80)

for _, row in candidate_errors.iterrows():
    print("Customer Message:", row["instruction"])
    print("True Intent:", row["true_intent"])
    print("Old Prediction:", row["old_prediction"])
    print("New Prediction:", row["new_prediction"])
    print("-" * 80)

Candidate Model Errors: 7
Customer Message: order cancellation
True Intent: cancel_order
Old Prediction: check_cancellation_fee
New Prediction: check_cancellation_fee
--------------------------------------------------------------------------------
Customer Message: i nmeed assistance removing some products
True Intent: change_order
Old Prediction: delete_account
New Prediction: delete_account
--------------------------------------------------------------------------------
Customer Message: can you help me changing a product of a purchase?
True Intent: change_order
Old Prediction: place_order
New Prediction: place_order
--------------------------------------------------------------------------------
Customer Message: hep changing some products of an order
True Intent: change_order
Old Prediction: place_order
New Prediction: place_order
--------------------------------------------------------------------------------
Customer Message: i dont know how to gt the invoices from {{Person Name}

In [146]:
import joblib
import json
from pathlib import Path

PROJECT_ROOT = Path.home() / "Desktop" / "Lumora"

MODEL_PATH = (
    PROJECT_ROOT / "models" / "lumora_intent_model_v2.joblib"
)

CONFIG_PATH = (
    PROJECT_ROOT / "data" / "processed"
    / "selected_intent_config_v2.json"
)

# Prepare model artifact
intent_artifact_v2 = {
    "vectorizer": candidate_vectorizer,
    "classifier": candidate_model,
    "intents": sorted(candidate_model.classes_.tolist())
}

# Save model
joblib.dump(intent_artifact_v2, MODEL_PATH)

# Save configuration and evaluation results
config = {
    "model_version": "v2",
    "model_type": "Character TF-IDF + LinearSVC",
    "ngram_range": [3, 5],
    "min_df": 2,
    "max_df": 0.95,
    "sublinear_tf": True,
    "C": 1,
    "augmentation_examples": len(augmentation_filtered),
    "validation_accuracy": candidate_accuracy,
    "validation_macro_f1": candidate_macro_f1,
    "test_accuracy": results["Candidate Model"]["Accuracy"],
    "test_macro_f1": results["Candidate Model"]["Macro F1"],
    "status": "selected_candidate"
}

CONFIG_PATH.write_text(
    json.dumps(config, indent=4),
    encoding="utf-8"
)

print("Intent Model V2 saved successfully.")
print("Model:", MODEL_PATH)
print("Configuration:", CONFIG_PATH)

Intent Model V2 saved successfully.
Model: /Users/raghad/Desktop/Lumora/models/lumora_intent_model_v2.joblib
Configuration: /Users/raghad/Desktop/Lumora/data/processed/selected_intent_config_v2.json


In [147]:
from src.retriever import retrieve_documents, retrieve_for_customer

query = "Where is my order?"

# البحث التقليدي
traditional_results = retrieve_documents(
    query=query,
    top_k=3
)

# البحث المحسّن باستخدام النية
intent_results = retrieve_for_customer(
    query=query,
    top_k=3
)

print("CUSTOMER QUERY:", query)
print("PREDICTED CATEGORY:", intent_results["category"])
print("PREDICTED INTENT:", intent_results["intent"])

print("\n--- TRADITIONAL RETRIEVAL ---")

for result in traditional_results:
    print(
        result["title"],
        "| Score:", result["score"]
    )

print("\n--- INTENT-AWARE RETRIEVAL ---")

for result in intent_results["documents"]:
    print(
        result["title"],
        "| Score:", result["score"]
    )

ImportError: cannot import name 'retrieve_for_customer' from 'src.retriever' (/Users/raghad/Desktop/Lumora/src/retriever.py)

In [148]:
import importlib
import src.retriever

importlib.reload(src.retriever)

from src.retriever import retrieve_documents, retrieve_for_customer

In [149]:
query = "Where is my order?"

In [150]:
from src.retriever import retrieve_documents, retrieve_for_customer

query = "Where is my order?"

print("Starting retrieval test...")

traditional_results = retrieve_documents(query, top_k=3)

intent_results = retrieve_for_customer(query, top_k=3)

print("\nCUSTOMER QUERY:", query)
print("CATEGORY:", intent_results["category"])
print("INTENT:", intent_results["intent"])

print("\n--- TRADITIONAL RETRIEVAL ---")

for result in traditional_results:
    print(result["title"], "| Score:", result["score"])

print("\n--- INTENT-AWARE RETRIEVAL ---")

for result in intent_results["documents"]:
    print(result["title"], "| Score:", result["score"])

print("\nTest completed.")

Starting retrieval test...

CUSTOMER QUERY: Where is my order?
CATEGORY: ORDER
INTENT: place_order

--- TRADITIONAL RETRIEVAL ---
2.2 Order Cancellation | Score: 0.2372
2.3 Order Modification | Score: 0.1852
2.1 Order Tracking | Score: 0.1818

--- INTENT-AWARE RETRIEVAL ---
2.2 Order Cancellation | Score: 0.2372
2.3 Order Modification | Score: 0.1852
2.1 Order Tracking | Score: 0.1818

Test completed.


In [151]:
import importlib

import src.intelligence
import src.retriever

# Reload intelligence first
importlib.reload(src.intelligence)

# Then reload retriever
importlib.reload(src.retriever)

from src.intelligence import analyze_message
from src.retriever import retrieve_documents, retrieve_for_customer

# Test the exact query
result = analyze_message("Where is my order?")

print("Category:", result["category"])
print("Intent:", result["intent"])

Category: ORDER
Intent: track_order


In [152]:
query = "Where is my order?"

traditional_results = retrieve_documents(
    query=query,
    top_k=3
)

intent_results = retrieve_for_customer(
    query=query,
    top_k=3
)

print("PREDICTED INTENT:", intent_results["intent"])

print("\n--- TRADITIONAL RETRIEVAL ---")
for result in traditional_results:
    print(result["title"], "| Score:", result["score"])

print("\n--- INTENT-AWARE RETRIEVAL ---")
for result in intent_results["documents"]:
    print(result["title"], "| Score:", result["score"])

PREDICTED INTENT: track_order

--- TRADITIONAL RETRIEVAL ---
2.2 Order Cancellation | Score: 0.2372
2.3 Order Modification | Score: 0.1852
2.1 Order Tracking | Score: 0.1818

--- INTENT-AWARE RETRIEVAL ---
2.1 Order Tracking | Score: 0.3419
4.3 Refund Tracking | Score: 0.1399
2.2 Order Cancellation | Score: 0.133


In [153]:
test_queries = [
    "How long does a refund take?",
    "Can I change my shipping address?",
    "What payment methods do you accept?",
    "I forgot my password",
    "I want to speak to a human agent"
]

for query in test_queries:

    result = retrieve_for_customer(query, top_k=3)

    print("\n" + "=" * 65)
    print("QUERY:", query)
    print("CATEGORY:", result["category"])
    print("INTENT:", result["intent"])

    print("\nTOP RETRIEVED DOCUMENTS:")

    for document in result["documents"]:
        print(
            document["title"],
            "| Score:",
            document["score"]
        )


QUERY: How long does a refund take?
CATEGORY: REFUND
INTENT: check_refund_policy

TOP RETRIEVED DOCUMENTS:
4.4 Refund Policy Questions | Score: 0.367
4.1 Return Eligibility | Score: 0.203
4.2 Refund Processing | Score: 0.1255

QUERY: Can I change my shipping address?
CATEGORY: SHIPPING
INTENT: change_shipping_address

TOP RETRIEVED DOCUMENTS:
3.2 Shipping Address Changes | Score: 0.4379
3.1 Delivery Options | Score: 0.1575
3.3 Delivery Delays | Score: 0.1082

QUERY: What payment methods do you accept?
CATEGORY: PAYMENT
INTENT: check_payment_methods

TOP RETRIEVED DOCUMENTS:
5.1 Accepted Payment Methods | Score: 0.3713
5.2 Payment Issues | Score: 0.0689
5.3 Payment Security | Score: 0.0623

QUERY: I forgot my password
CATEGORY: ACCOUNT
INTENT: recover_password

TOP RETRIEVED DOCUMENTS:
6.2 Password Recovery | Score: 0.312
6.1 Account Registration | Score: 0.137

QUERY: I want to speak to a human agent
CATEGORY: CONTACT
INTENT: contact_human_agent

TOP RETRIEVED DOCUMENTS:
8.2 Human Age

In [154]:
import importlib
import src.intelligence
import src.retriever

importlib.reload(src.intelligence)
importlib.reload(src.retriever)

from src.retriever import retrieve_documents, retrieve_for_customer

In [155]:
test_queries = [
    "Where is my order?",
    "How long does a refund take?",
    "Can I change my shipping address?",
    "What payment methods do you accept?",
    "I forgot my password",
    "I want to speak to a human agent"
]

for query in test_queries:
    print("=" * 80)
    print(f"QUERY: {query}")

    result = retrieve_for_customer(query, top_k=3)

    print(f"Category: {result['category']}")
    print(f"Intent: {result['intent']}")

    print("\nTop Retrieved Documents:")

    for rank, doc in enumerate(result["documents"], start=1):
        print(
            f"{rank}. {doc['title']} | "
            f"Final: {doc['score']:.4f} | "
            f"Original: {doc['original_score']:.4f} | "
            f"Intent: {doc['intent_score']:.4f}"
        )

    print()

QUERY: Where is my order?
Category: ORDER
Intent: track_order

Top Retrieved Documents:
1. 2.1 Order Tracking | Final: 0.2298 | Original: 0.1818 | Intent: 0.3419
2. 2.2 Order Cancellation | Final: 0.2059 | Original: 0.2372 | Intent: 0.1330
3. 2.3 Order Modification | Final: 0.1623 | Original: 0.1852 | Intent: 0.1090

QUERY: How long does a refund take?
Category: REFUND
Intent: check_refund_policy

Top Retrieved Documents:
1. 4.4 Refund Policy Questions | Final: 0.1785 | Original: 0.0978 | Intent: 0.3670
2. 4.2 Refund Processing | Final: 0.1515 | Original: 0.1626 | Intent: 0.1255
3. 4.3 Refund Tracking | Final: 0.1088 | Original: 0.1096 | Intent: 0.1068

QUERY: Can I change my shipping address?
Category: SHIPPING
Intent: change_shipping_address

Top Retrieved Documents:
1. 3.2 Shipping Address Changes | Final: 0.4122 | Original: 0.4013 | Intent: 0.4379
2. 3.1 Delivery Options | Final: 0.1120 | Original: 0.0925 | Intent: 0.1575
3. 6.1 Account Registration | Final: 0.0786 | Original: 0.08

In [156]:
query = "How long does a refund take?"

print("WITHOUT INTENT")
for doc in retrieve_documents(query, top_k=3):
    print(doc["title"], doc["score"])

print("\nWITH INTENT")
result = retrieve_for_customer(query, top_k=3)

for doc in result["documents"]:
    print(doc["title"], doc["score"])

WITHOUT INTENT
4.2 Refund Processing 0.1626
4.3 Refund Tracking 0.1096
4.4 Refund Policy Questions 0.0978

WITH INTENT
4.4 Refund Policy Questions 0.1785
4.2 Refund Processing 0.1515
4.3 Refund Tracking 0.1088


In [157]:
queries = [
    "Can I change my shipping address?",
    "What payment methods do you accept?",
    "I forgot my password",
    "I want to speak to a human agent"
]

for query in queries:
    print("=" * 70)
    print("QUERY:", query)

    for doc in retrieve_documents(query, top_k=3):
        print(f"{doc['title']} | Score: {doc['score']:.4f}")

QUERY: Can I change my shipping address?
3.2 Shipping Address Changes | Score: 0.4013
3.1 Delivery Options | Score: 0.0925
6.1 Account Registration | Score: 0.0811
QUERY: What payment methods do you accept?
5.1 Accepted Payment Methods | Score: 0.3713
5.2 Payment Issues | Score: 0.0689
5.3 Payment Security | Score: 0.0623
QUERY: I forgot my password
6.2 Password Recovery | Score: 0.3120
6.1 Account Registration | Score: 0.1370
QUERY: I want to speak to a human agent
8.2 Human Agent Escalation | Score: 0.3979
8.1 Contacting Support | Score: 0.0707
6.2 Password Recovery | Score: 0.0441


In [158]:
query = "How long does a refund take?"

result = retrieve_for_customer(query, top_k=3)

print("Detected intent:", result["intent"])

for doc in result["documents"]:
    print(
        doc["title"],
        "| Original:", doc["original_score"],
        "| Intent:", doc["intent_score"],
        "| Hybrid:", doc["score"]
    )

Detected intent: check_refund_policy
4.4 Refund Policy Questions | Original: 0.0978 | Intent: 0.367 | Hybrid: 0.1785
4.2 Refund Processing | Original: 0.1626 | Intent: 0.1255 | Hybrid: 0.1515
4.3 Refund Tracking | Original: 0.1096 | Intent: 0.1068 | Hybrid: 0.1088


In [159]:
query = "How long does a refund take?"

documents = retrieve_documents(query, top_k=22)

original_scores = {
    doc["chunk_id"]: doc["score"]
    for doc in documents
}

result = retrieve_for_customer(query, top_k=22)

intent_scores = {
    doc["chunk_id"]: doc["intent_score"]
    for doc in result["documents"]
}

for weight in [0.1, 0.2, 0.3, 0.4, 0.5]:
    ranked = sorted(
        documents,
        key=lambda doc: (
            (1 - weight) * original_scores[doc["chunk_id"]]
            + weight * intent_scores.get(
                doc["chunk_id"],
                original_scores[doc["chunk_id"]]
            )
        ),
        reverse=True
    )

    print(f"\nIntent Weight: {weight}")
    for doc in ranked[:3]:
        print(doc["title"])


Intent Weight: 0.1
4.2 Refund Processing
4.4 Refund Policy Questions
4.3 Refund Tracking

Intent Weight: 0.2
4.2 Refund Processing
4.4 Refund Policy Questions
4.3 Refund Tracking

Intent Weight: 0.3
4.4 Refund Policy Questions
4.2 Refund Processing
4.3 Refund Tracking

Intent Weight: 0.4
4.4 Refund Policy Questions
4.2 Refund Processing
4.3 Refund Tracking

Intent Weight: 0.5
4.4 Refund Policy Questions
4.2 Refund Processing
4.3 Refund Tracking


In [160]:
queries = [
    "Where is my order?",
    "How long does a refund take?",
    "Can I change my shipping address?",
    "What payment methods do you accept?",
    "I forgot my password",
    "I want to speak to a human agent"
]

intent_weight = 0.2

for query in queries:
    result = retrieve_for_customer(query, top_k=22)

    documents = result["documents"]

    ranked = sorted(
        documents,
        key=lambda doc: (
            (1 - intent_weight) * doc["original_score"]
            + intent_weight * doc["intent_score"]
        ),
        reverse=True
    )

    print("\n" + "=" * 70)
    print("QUERY:", query)
    print("INTENT:", result["intent"])

    for rank, doc in enumerate(ranked[:3], start=1):
        print(f"{rank}. {doc['title']}")


QUERY: Where is my order?
INTENT: track_order
1. 2.2 Order Cancellation
2. 2.1 Order Tracking
3. 2.3 Order Modification

QUERY: How long does a refund take?
INTENT: check_refund_policy
1. 4.2 Refund Processing
2. 4.4 Refund Policy Questions
3. 4.3 Refund Tracking

QUERY: Can I change my shipping address?
INTENT: change_shipping_address
1. 3.2 Shipping Address Changes
2. 3.1 Delivery Options
3. 6.1 Account Registration

QUERY: What payment methods do you accept?
INTENT: check_payment_methods
1. 5.1 Accepted Payment Methods
2. 5.2 Payment Issues
3. 5.3 Payment Security

QUERY: I forgot my password
INTENT: recover_password
1. 6.2 Password Recovery
2. 6.1 Account Registration

QUERY: I want to speak to a human agent
INTENT: contact_human_agent
1. 8.2 Human Agent Escalation
2. 8.1 Contacting Support
3. 6.2 Password Recovery


In [161]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

intent_artifact = load_intent_model()

X_val_intent = val_df["instruction"].astype(str)
y_val_intent = val_df["intent"]

X_val_vectorized = intent_artifact["vectorizer"].transform(
    X_val_intent
)

y_val_v2_pred = intent_artifact["classifier"].predict(
    X_val_vectorized
)

print("V2 Validation Accuracy:",
      accuracy_score(y_val_intent, y_val_v2_pred))

print("V2 Validation Macro F1:",
      f1_score(y_val_intent, y_val_v2_pred, average="macro"))

print("\nClassification Report:")
print(classification_report(
    y_val_intent,
    y_val_v2_pred,
    zero_division=0
))

V2 Validation Accuracy: 0.9987626825043306
V2 Validation Macro F1: 0.9987305680239265

Classification Report:
                          precision    recall  f1-score   support

            cancel_order       1.00      1.00      1.00       163
            change_order       0.99      0.99      0.99       125
 change_shipping_address       0.99      1.00      1.00       144
  check_cancellation_fee       1.00      1.00      1.00       150
           check_invoice       1.00      1.00      1.00       130
   check_payment_methods       1.00      1.00      1.00       153
     check_refund_policy       1.00      0.99      1.00       160
               complaint       1.00      1.00      1.00       137
contact_customer_service       1.00      1.00      1.00       149
     contact_human_agent       1.00      1.00      1.00       146
          create_account       0.99      1.00      1.00       186
          delete_account       1.00      1.00      1.00       151
        delivery_options       

In [162]:
import pandas as pd

validation_errors = pd.DataFrame({
    "instruction": X_val_intent,
    "actual_intent": y_val_intent,
    "predicted_intent": y_val_v2_pred
})

validation_errors = validation_errors[
    validation_errors["actual_intent"]
    != validation_errors["predicted_intent"]
]

print("Number of validation errors:", len(validation_errors))

display(validation_errors)

Number of validation errors: 5


,instruction,actual_intent,predicted_intent
1443,could you help me to delete something?,change_order,place_order
6860,am I entitled to a reimbursement?,check_refund_policy,get_refund
21679,assistance opening user account,registration_problems,create_account
23556,deliveryaddress issue,set_up_shipping_address,change_shipping_address
25534,where is purechase {{Order Number}},track_order,change_order


In [163]:
%whos

Variable                    Type                  Data/Info
-----------------------------------------------------------
BALANCED_ERRORS_PATH        PosixPath             /Users/raghad/Desktop/Lum<...>_validation_errors_v1.csv
CONFIG_DIR                  PosixPath             /Users/raghad/Desktop/Lumora/data/processed
CONFIG_PATH                 PosixPath             /Users/raghad/Desktop/Lum<...>ted_intent_config_v2.json
Counter                     type                  <class 'collections.Counter'>
DATA_PATH                   PosixPath             /Users/raghad/Desktop/Lum<...>tomer_support_tickets.csv
ERRORS_PATH                 PosixPath             /Users/raghad/Desktop/Lum<...>_validation_errors_v1.csv
FeatureUnion                ABCMeta               <class 'sklearn.pipeline.FeatureUnion'>
LinearSVC                   type                  <class 'sklearn.svm._classes.LinearSVC'>
LogisticRegression          type                  <class 'sklearn.linear_mo<...>stic.LogisticRegressi

In [164]:
variables_to_check = [
    "X_train",
    "y_train",
    "X_train_candidate",
    "y_train_candidate",
    "X_val",
    "y_val",
    "X_test",
    "y_test"
]

for name in variables_to_check:
    if name in globals():
        value = globals()[name]

        print(f"\n{name}")
        print("Type:", type(value))
        print("Shape:", getattr(value, "shape", "No shape"))
        print("Length:", len(value))
    else:
        print(f"\n{name}: Not found")


X_train
Type: <class 'pandas.Series'>
Shape: (18798,)
Length: 18798

y_train
Type: <class 'pandas.Series'>
Shape: (18798,)
Length: 18798

X_train_candidate
Type: <class 'pandas.Series'>
Shape: (18810,)
Length: 18810

y_train_candidate
Type: <class 'pandas.Series'>
Shape: (18810,)
Length: 18810

X_val
Type: <class 'pandas.Series'>
Shape: (4041,)
Length: 4041

y_val
Type: <class 'pandas.Series'>
Shape: (4041,)
Length: 4041

X_test
Type: <class 'pandas.Series'>
Shape: (4033,)
Length: 4033

y_test
Type: <class 'pandas.Series'>
Shape: (4033,)
Length: 4033


In [165]:
print("Original training size:", len(X_train))
print("Candidate training size:", len(X_train_candidate))

print("\nAdditional training examples:")
additional_examples = X_train_candidate.loc[
    ~X_train_candidate.index.isin(X_train.index)
]

display(
    pd.DataFrame({
        "instruction": additional_examples,
        "intent": y_train_candidate.loc[additional_examples.index]
    })
)

Original training size: 18798
Candidate training size: 18810

Additional training examples:


,instruction,intent
0,i have a question about cancelling oorder {{Or...,cancel_order
6,I have got to cancel purchase {{Order Number}},cancel_order
15,need help with canceling order {{Order Number}},cancel_order
19,need help with canceling purchase {{Order Numb...,cancel_order
27,want assistance to cancel order {{Order Number}},cancel_order
...,...,...
18788,I don't know how I could track the compensation,track_refund
18796,I don't know what to do to see my reimbursemen...,track_refund
18800,I'm trying to locate my purchase,track_order
18802,Could you tell me the current status of my pur...,track_order


In [166]:
keywords = [
    "X_train_candidate",
    "y_train_candidate",
    "intent_model_v2",
    "LinearSVC",
    ".fit("
]

for cell_number, cell in enumerate(In):
    if any(keyword in cell for keyword in keywords):
        print(f"\n{'=' * 70}")
        print(f"CELL: {cell_number}")
        print(cell)


CELL: 11
# Initialize the baseline Logistic Regression model

baseline_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# Train the model using training data only

baseline_model.fit(X_train_tfidf, y_train)

print("Baseline Logistic Regression trained successfully.")

CELL: 20
# Initialize a class-balanced Logistic Regression model

balanced_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

# Train the balanced model

balanced_model.fit(X_train_tfidf, y_train)

print("Balanced Logistic Regression trained successfully.")

CELL: 31
# Train Logistic Regression with Trigram TF-IDF

trigram_balanced_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

trigram_balanced_model.fit(
    X_train_tfidf_trigram,
    y_train
)

print("Trigram Balanced Model trained successfully.")

CELL: 37
# Train Logistic Regression with C=0.5

tuned_model_c05.fit(
    X_train_tfidf,
    y_train
)


In [167]:
print("===== V2 MODEL METADATA =====")

print("Training intents:")
print(artifact["intents"])

print("\n===== MODEL PARAMETERS =====")

print(classifier.get_params())

print("\n===== MODEL DIMENSIONS =====")

print("Number of classes:", len(classifier.classes_))
print("Number of features:", classifier.coef_.shape[1])
print("Coefficient shape:", classifier.coef_.shape)

===== V2 MODEL METADATA =====
Training intents:
['cancel_order', 'change_order', 'change_shipping_address', 'check_cancellation_fee', 'check_invoice', 'check_payment_methods', 'check_refund_policy', 'complaint', 'contact_customer_service', 'contact_human_agent', 'create_account', 'delete_account', 'delivery_options', 'delivery_period', 'edit_account', 'get_invoice', 'get_refund', 'newsletter_subscription', 'payment_issue', 'place_order', 'recover_password', 'registration_problems', 'review', 'set_up_shipping_address', 'switch_account', 'track_order', 'track_refund']

===== MODEL PARAMETERS =====
{'C': 1, 'class_weight': None, 'dual': 'auto', 'fit_intercept': True, 'intercept_scaling': 1, 'loss': 'squared_hinge', 'max_iter': 5000, 'multi_class': 'ovr', 'penalty': 'l2', 'random_state': 42, 'tol': 0.0001, 'verbose': 0}

===== MODEL DIMENSIONS =====
Number of classes: 27
Number of features: 9166
Coefficient shape: (27, 9166)


In [168]:
for i, cell in enumerate(In):
    if "fit(" in cell and ("candidate" in cell or "intent" in cell):
        print(f"\n===== CELL {i} =====\n")
        print(cell)


===== CELL 126 =====

# Intent Classification: Character TF-IDF + Linear SVM

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report

intent_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_intent_vec = intent_vectorizer.fit_transform(X_train_intent)
X_val_intent_vec = intent_vectorizer.transform(X_val_intent)

intent_model = LinearSVC(
    C=1,
    random_state=42,
    max_iter=5000
)

intent_model.fit(X_train_intent_vec, y_train_intent)

y_val_intent_pred = intent_model.predict(X_val_intent_vec)

print("Intent Model Evaluation")
print("-" * 40)

print("Accuracy:", accuracy_score(y_val_intent, y_val_intent_pred))
print("Macro F1:", f1_score(y_val_intent, y_val_intent_pred, average="macro"))
print("Weighted F1:", f1_score(y_val_intent, y_val_intent_pred, average="weighted"))

prin

In [169]:
print("Training data size:", len(X_train_intent))
print("Training labels size:", len(y_train_intent))

print("\nFirst 5 training examples:")
display(pd.DataFrame({
    "instruction": X_train_intent.head(),
    "intent": y_train_intent.head()
}))

Training data size: 18798
Training labels size: 18798

First 5 training examples:


,instruction,intent
1,i have a question about cancelling oorder {{Or...,cancel_order
2,i need help cancelling puchase {{Order Number}},cancel_order
3,I need to cancel purchase {{Order Number}},cancel_order
4,"I cannot afford this order, cancel purchase {{...",cancel_order
5,can you help me cancel order {{Order Number}}?,cancel_order


In [170]:
print("\n".join(
    line for line in In[126].splitlines()
    if ".fit(" in line or "joblib.dump" in line or "intent_model" in line
))

intent_model = LinearSVC(
intent_model.fit(X_train_intent_vec, y_train_intent)
y_val_intent_pred = intent_model.predict(X_val_intent_vec)


In [171]:
print("Saved V2 model classes:", len(classifier.classes_))
print("Original model classes:", len(intent_model.classes_))

print(
    "Are their coefficients identical?",
    np.array_equal(classifier.coef_, intent_model.coef_)
)

Saved V2 model classes: 27
Original model classes: 27
Are their coefficients identical? True


In [172]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

robustness_queries = [
    "I need to change my order.",
    "Can I change an order after placing it?",
    "How can I edit my purchase?",
    "Where is my package?",
    "My payment is not working.",
    "Can you tell me something about your store?",
    "What are your opening hours?",
    "I need help with something."
]

training_matrix = intent_vectorizer.transform(X_train_intent)

similarity_results = []

for query in robustness_queries:

    query_vector = intent_vectorizer.transform([query])

    similarities = cosine_similarity(
        query_vector,
        training_matrix
    ).flatten()

    best_index = similarities.argmax()

    similarity_results.append({
        "query": query,
        "nearest_training_similarity": round(
            float(similarities[best_index]), 4
        ),
        "nearest_training_example": X_train_intent.iloc[best_index],
        "nearest_training_intent": y_train_intent.iloc[best_index]
    })

similarity_df = pd.DataFrame(similarity_results)

display(similarity_df)

,query,nearest_training_similarity,nearest_training_example,nearest_training_intent
0,I need to change my order.,0.7888,change order,change_order
1,Can I change an order after placing it?,0.6112,change order,change_order
2,How can I edit my purchase?,0.6253,where do I make a purchase?,place_order
3,Where is my package?,0.7510,where do I check how soon can I expect my pack...,delivery_period
4,My payment is not working.,0.4809,payment help,payment_issue
5,Can you tell me something about your store?,0.5126,can ya help me order something,place_order
6,What are your opening hours?,0.4990,opening pro account,create_account
7,I need help with something.,0.7304,help me to buyy something,place_order


In [173]:
saved_vectorizer = artifact["vectorizer"]

print(
    "Same vocabulary:",
    saved_vectorizer.vocabulary_ == intent_vectorizer.vocabulary_
)

print(
    "Same IDF:",
    np.array_equal(
        saved_vectorizer.idf_,
        intent_vectorizer.idf_
    )
)

print(
    "Same feature count:",
    len(saved_vectorizer.vocabulary_) == len(intent_vectorizer.vocabulary_)
)

Same vocabulary: True
Same IDF: True
Same feature count: True
